# Phase 3 - Exploratory Data Analysis
## Patterns of Recorded Civilian Targeting in Gaza

**DS312 | Group 7 | Abdula, Aque, Bautro, Budlong, Mashburn, Orano, Pandi, Prollo**

Human-rights documentation teams need to decide which places, periods and incident groups deserve closer investigation. This notebook uses ACLED's recorded events to describe recurring patterns and identify useful starting points for that review; it does not determine intent, responsibility or legal classification.

**Scope.** We study Gaza-coded records from **January 1, 2022 to September 30, 2025**, from the snapshot downloaded October 18, 2026. Phase 1 originally included the West Bank, but Phase 2 narrowed the working study to Gaza after consultation. One row is one coded event record, which can summarize several reported actions; it is not necessarily one attack or one victim.

**How to read this report.** First we explain the data, then describe individual fields, compare groups, test two questions, and examine unusual weeks and geographic concentrations. Readers looking for the answers can go to [Task 6](#Task-6---Answers-to-the-business-questions); the supporting figures and tables are linked there, followed by the [modeling plan](#Task-7---Key-insights-and-modeling-plan).

| Question | What this Phase 3 report answers |
|---|---|
| BQ1 | How do recorded focal counts and shares differ before and from October 7, 2023? |
| BQ2 | Which recorded actors and event types recur among focal events? |
| BQ3 | Where are post-cutoff land focal events concentrated, and what changes with location precision? |
| BQ4 | Which district-weeks have high or sustained activity relative to recent local history? |
| BQ5 | Which findings change with periods, precision restrictions or analytical settings? |

The before/after comparison is descriptive. Temporal and spatial previews use the post-October 7 period; West Bank comparisons, verified actor roles and a full reporting-bias sensitivity study are outside this report.


## Reading guide and rubric coverage

| Task | Evidence in this notebook |
|---|---|
| [1. Describe the data](#Task-1---What-does-the-dataset-contain?) | Profiles, distributions and category counts |
| [2. Compare groups](#Task-2---How-do-groups-differ?) | Rates, severity, actor cross-tabs, district/weekly comparisons and association matrices |
| [3. Visualization register](#Task-3---Main-figure-register) | Links to the figures; each figure stays beside its interpretation |
| [4. Test two hypotheses](#Task-4---Test-the-period-and-event-type-differences) | Period and event-type tests, uncertainty intervals and sensitivity checks |
| [5. Find patterns](#Task-5---Which-patterns-deserve-closer-review?) | Imbalance, unusual values/weeks, trends, calendar patterns, persistence and clusters |
| [6. Answer the questions](#Task-6---Answers-to-the-business-questions) | BQ1-BQ5 with exact evidence links |
| [7. Plan the next phase](#Task-7---Key-insights-and-modeling-plan) | Seven key insights and a proposed clustering/anomaly-detection plan |

**Three quantities that must not be confused:** a **count** is a number of records, a **rate** adjusts a count for time observed, and a **share** divides one group of records by a stated total. Larger counts do not automatically imply larger shares.


## Data guide - what is being counted?

**Focal event** means an ACLED `Civilian targeting` flag **and** the exact label `Civilians (Palestine)` in either actor field. This project rule selects records for review, not all civilian harm. **Political violence** means that exact token appears in `disorder_type`, including records carrying more than one disorder label. Every focal record in this snapshot belongs to that political-violence group.

**Non-focal** means a record does not meet both focal conditions. In focal-versus-non-focal comparisons, we first restrict to political-violence records, so non-focal means the other political records in that comparison group; it does not mean civilians were unharmed.

**Focal share** is focal events divided by political-violence records in the same group, place and period. A blank targeting label is retained in the source and means the flag was not recorded; it is not proof that civilians were unharmed.

### The four Phase 2 outputs

All files are loaded once from `data/processed/`; later sections name their source. These are related views of the same snapshot, not four independent datasets to add together.

| File / dataframe | One row and scope | Used for |
|---|---|---|
| `acled_events_clean.csv` / `master` | One Gaza-coded event; 26,664 rows, 43 columns; both periods | Distributions, period/actor/type comparisons and tests |
| `acled_gaza_week.csv` / `gaza_week` | One Gaza-wide Saturday-Friday week after the cutoff; 104 rows, 13 columns | Counts, shares, history, trends and groups of weeks |
| `acled_district_week.csv` / `district_week` | One known district and week; five districts x 104 weeks; 14 columns | District comparisons and local anomaly screening |
| `acled_focal_incidents.csv` / `incidents` | One post-cutoff land focal event; 10,583 rows, 45 columns | Spatial concentration, precision checks and DBSCAN |

There are **10,714 focal events across both periods**, including **10,651 from October 7 onward**. Four Gaza records lack a district (three focal), so district-week focal counts total **10,648**. The spatial file instead removes **68 post-cutoff focal records** with a location starting `Coast of `, leaving **10,583**; coastal records remain in Gaza-wide and known-district temporal totals. The master contains 701 coast-coded records across all types and periods.

### Preparation a reader needs to know

- Phase 2 retained original actor labels, structural blanks and reported fatality totals, and dropped the entirely empty `admin3` field. It did not impute actors, trim fatalities or delete repeated coordinates.
- `period` splits at **October 7, 2023**, included in the later period: **644 days before, 725 days from the cutoff**. `month_start` groups event calendar months; `week_start` begins on Saturday.
- The weekly calendar keeps zero-count weeks. The last week starts **September 27, 2025**, but observation ends September 30: it is partial, shown where useful and excluded from complete-week comparisons.
- `focal_share` is undefined when the political count is zero. `history_eligible` requires both a complete week and a positive political count; `focal_count_for_history` masks other weeks.
- `previous_week_focal` is the preceding calendar week's eligible count. `previous_four_weeks_focal` is the **sum of the preceding four calendar weeks**, shifted before summing; it is missing if any required history is unavailable. Both features are saved in the two weekly CSVs.
- `geo_precision` and `time_precision` are ordered source quality codes, not distances or measurements. Code 1 is the most precise category; a `geo_precision_1` filter keeps only that category, without making the coordinates exact incident points.

### Logs and review flags

**Inherited event log:** `fatalities_log1p`, saved in the event master and spatial file, is **ln(1 + reported event fatalities)**. Phase 2 created it so a long tail can be displayed more clearly while zero stays zero; for example, a raw count of 0 becomes 0 and 9 becomes about 2.30. It is not a count of civilian deaths.

**Inherited review flag:** `high_fatalities_review` marks fatalities **greater than 23**, the 99th percentile of the full Palestine raw export, not the Gaza-only percentile (39). A flag calls for checking the source; it does not remove or invalidate the record.

**New weekly logs:** the [grouping-of-weeks preview](#Grouping-complete-weeks-by-recorded-activity) creates log1p focal and political counts from `acled_gaza_week.csv` in Phase 3, then standardizes them for K-means. These are different from the inherited fatality log. A logarithmic axis or color scale elsewhere changes only how numbers are displayed.

**Runtime:** run from this project folder with NumPy, pandas, Matplotlib, seaborn, SciPy (with `MonteCarloMethod`) and scikit-learn installed. The setup restores dates and precision categories because CSV files do not preserve these types.


In [ ]:
# Set thread controls before scikit-learn imports (avoids Windows worker warnings).
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
import warnings
warnings.filterwarnings("ignore", message=".*physical cores.*")
warnings.filterwarnings("ignore", message=".*memory leak on Windows.*")
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")  # Save figures as visible notebook outputs when run top-to-bottom.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy.stats import chi2_contingency, MonteCarloMethod, binomtest, fisher_exact, spearmanr
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10, "legend.fontsize": 9})
plt.rcParams["axes.grid.axis"] = "y"
SEED, ALPHA, N_RESAMPLES, Z = 42, 0.05, 99_999, 1.96
np.random.seed(SEED)
C_FOCAL, C_OTHER, C_GREY, C_WARN = "#8D3B42", "#315A7D", "#9AA5B1", "#E0A458"

def find_data_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "data" / "processed"
        if (candidate / "acled_events_clean.csv").is_file(): return candidate
    raise FileNotFoundError("Cannot find data/processed/acled_events_clean.csv. Run from the project root or set DATA_DIR.")

DATA_DIR = find_data_dir()
master = pd.read_csv(DATA_DIR / "acled_events_clean.csv", parse_dates=["event_date", "month_start", "week_start"])
for col in ("geo_precision", "time_precision"):
    master[col] = pd.Categorical(master[col], categories=[1, 2, 3], ordered=True)
gaza_week = pd.read_csv(DATA_DIR / "acled_gaza_week.csv", parse_dates=["week_start", "week_end"])
district_week = pd.read_csv(DATA_DIR / "acled_district_week.csv", parse_dates=["week_start", "week_end"])
incidents = pd.read_csv(DATA_DIR / "acled_focal_incidents.csv", parse_dates=["event_date", "month_start", "week_start"])
for col in ("geo_precision", "time_precision"):
    incidents[col] = pd.Categorical(incidents[col], categories=[1, 2, 3], ordered=True)
# Alternate weekly names used by the profiling tables refer to the same loaded data.
gaza_weekly, district_weekly = gaza_week, district_week
BLUE, GREEN, RED, GREY = "#315A7D", "#4B8A78", "#8D3B42", "#8A8F98"
cutoff = pd.Timestamp("2023-10-07")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
BEFORE, AFTER = "Before Oct 7, 2023", "Oct 7, 2023 onward"
CUTOFF = pd.Timestamp("2023-10-07")
COVERAGE_START, COVERAGE_END = pd.Timestamp("2022-01-01"), pd.Timestamp("2025-09-30")
PERIOD_DAYS = {BEFORE: (CUTOFF-COVERAGE_START).days, AFTER: (COVERAGE_END-CUTOFF).days+1}
EVENT_TYPES = ["Explosions/Remote violence", "Violence against civilians", "Battles", "Protests", "Riots"]
political = master.loc[master.political_violence].copy()
print("Phase 2 directory:", DATA_DIR.resolve())
print("master:",master.shape,"| gaza_week:",gaza_week.shape,"| district_week:",district_week.shape,"| incidents:",incidents.shape)
print("Date coverage:",COVERAGE_START.date(),"to",COVERAGE_END.date())

from io import BytesIO
from IPython.display import Image as NotebookImage
supporting_figures = {}

def save_supporting_figure(key, figure):
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=140, bbox_inches="tight", facecolor="white")
    supporting_figures[key] = buffer.getvalue()
    plt.close(figure)

def show_supporting_figure(key):
    display(NotebookImage(data=supporting_figures[key]))


### Check the snapshot before interpreting it

The table below checks the four inputs together, including their dates, scope, keys and totals. A failed check stops the notebook so the fixed narrative cannot silently describe a different dataset.


In [ ]:
checks = pd.DataFrame([
 ("Event master",(26664,43),master.shape),("Gaza-week",(104,13),gaza_week.shape),
 ("District-week",(520,14),district_week.shape),("Land focal incidents",(10583,45),incidents.shape),
 ("Focal events",10714,int(master.focal_event.sum())),
 ("Focal before cutoff",63,int(master.loc[master.period==BEFORE,"focal_event"].sum())),
 ("Focal from cutoff",10651,int(master.loc[master.period==AFTER,"focal_event"].sum())),
 ("Political records before",389,int((political.period==BEFORE).sum())),
 ("Political records from cutoff",22364,int((political.period==AFTER).sum())),
 ("Missing districts",4,int(master.admin2.isna().sum())),
 ("Gaza-week focal total",10651,int(gaza_week.focal_events.sum())),
 ("District-week focal total",10648,int(district_week.focal_events.sum())),
 ("Incomplete Gaza weeks",1,int((~gaza_week.complete_week).sum())),
 ("Precision-1 land focal",7576,int(incidents.geo_precision_1.sum())),
 ("Period labels disagree with event dates",0,int((master.period != np.where(master.event_date.ge(CUTOFF), AFTER, BEFORE)).sum())),
 ("Duplicate event IDs",0,int(master.event_id_cnty.duplicated().sum())),
 ("Duplicate Gaza-week keys",0,int(gaza_week.duplicated(["admin1", "week_start"]).sum())),
 ("Duplicate district-week keys",0,int(district_week.duplicated(["admin2", "week_start"]).sum())),
 ("Focal records outside political violence",0,int((master.focal_event & ~master.political_violence).sum())),
 ("Unexpected political event types",0,int((~political.event_type.isin(EVENT_TYPES)).sum())),
 ("Political event-type count",5,int(political.event_type.nunique())),
 ("Coverage starts",COVERAGE_START,master.event_date.min()),
 ("Coverage ends",COVERAGE_END,master.event_date.max()),
],columns=["Check","Phase 2 reference","Loaded value"])
checks["Match"]=[a==b for a,b in zip(checks["Phase 2 reference"],checks["Loaded value"])]
display(checks)
assert checks.Match.all(), "Input snapshot differs: reconcile the checks before interpreting results."
print("All snapshot checks match.")
print("Incomplete week starts:",gaza_week.loc[~gaza_week.complete_week,"week_start"].dt.date.tolist())

assert np.allclose(master.fatalities_log1p, np.log1p(master.fatalities))
assert master.focal_event.equals(master.targeting_flagged & (master.actor1.eq("Civilians (Palestine)") | master.actor2.eq("Civilians (Palestine)")))
assert master.political_violence.equals(master.disorder_type.str.split(";").apply(lambda labels: "Political violence" in [x.strip() for x in labels]))
assert master.high_fatalities_review.equals(master.fatalities.gt(23))
assert incidents.event_id_cnty.tolist() == master.loc[master.land_spatial_eligible, "event_id_cnty"].tolist()
assert np.allclose(incidents[["latitude_rad", "longitude_rad"]], np.radians(incidents[["latitude", "longitude"]]))
for weekly, area in [(gaza_week, "admin1"), (district_week, "admin2")]:
    assert weekly.history_eligible.equals(weekly.complete_week & weekly.political_events.gt(0))
    assert np.allclose(weekly.focal_share, weekly.focal_events / weekly.political_events.where(weekly.political_events.gt(0)), equal_nan=True)
    for _, part in weekly.sort_values("week_start").groupby(area):
        history = part.focal_events.where(part.history_eligible)
        assert np.allclose(part.focal_count_for_history, history, equal_nan=True)
        assert np.allclose(part.previous_week_focal, history.shift(1), equal_nan=True)
        assert np.allclose(part.previous_four_weeks_focal, history.shift(1).rolling(4).sum(), equal_nan=True)
print("Focal rule, fatality log/flag, spatial membership/radians and eligible lag features verified.")


# Task 1 - What does the dataset contain?

Before comparing groups, we establish what a typical record looks like and where the data are uneven. This helps explain why later analyses use counts, medians, explicit denominators and quality restrictions.


The event master supplies event-level summaries, the weekly CSVs supply weekly profiles, and the spatial file is used only where stated. Tables provide detail; the short interpretation after each figure gives the main result and its limit.


**Why these distribution checks?** For count variables we inspect common values, zero frequency, percentiles and tails; categorical variables are described by counts, proportions, missingness and rare labels. Skewness, kurtosis and normality tests are optional diagnostics, not required steps for every field: the planned DBSCAN, Isolation Forest and median/MAD methods do not require normally distributed inputs, although skew and extreme values still affect transformations, distances and thresholds.


## 1.1 Field profiles and missing information

**Sources:** all columns of the event master and both weekly files; selected spatial/precision columns of the incident file. The tables report missing values and suitable summaries: dates get ranges, categories get their most common label, and numbers get minimum, median, mean and maximum.

The **median** is the middle value; the **mean** is the arithmetic average. A field with one constant value adds no variation here, and a blank value is not automatically a data-entry error.


In [ ]:
def profile_table(frame, categorical_columns=None, timestamp_columns=None):
    """Profile columns with summaries appropriate to their meaning and dtype."""
    categorical_columns = set(categorical_columns or [])
    timestamp_columns = set(timestamp_columns or [])
    rows = []

    for col in frame.columns:
        s = frame[col]
        n_missing = int(s.isna().sum())
        distinct = int(s.nunique(dropna=True))
        non_missing = int(s.notna().sum())

        if col in timestamp_columns:
            dt = pd.to_datetime(s, unit="s", errors="coerce", utc=True)
            kind = "record timestamp (UTC)"
            summary = (f"{dt.min().date()} to {dt.max().date()}"
                       if dt.notna().any() else "no valid timestamps")
        elif col in categorical_columns:
            kind = "identifier / category"
            if non_missing == 0:
                summary = "all values missing"
            elif distinct == non_missing:
                summary = "unique in every non-missing row"
            else:
                counts = s.value_counts(dropna=True)
                summary = f"mode = {str(counts.index[0])} ({counts.iloc[0] / non_missing:.1%})"
        elif pd.api.types.is_bool_dtype(s):
            kind = "boolean"
            summary = f"True = {s.mean():.1%}" if non_missing else "all values missing"
        elif isinstance(s.dtype, pd.CategoricalDtype):
            kind = "ordered category"
            counts = s.value_counts(dropna=True)
            summary = (f"mode = {counts.index[0]} ({counts.iloc[0] / non_missing:.1%})"
                       if non_missing else "all values missing")
        elif pd.api.types.is_datetime64_any_dtype(s):
            kind = "date"
            summary = f"{s.min().date()} to {s.max().date()}" if non_missing else "all values missing"
        elif pd.api.types.is_numeric_dtype(s):
            kind = "numeric"
            summary = (f"min {s.min():.6g} | median {s.median():.6g} | "
                       f"mean {s.mean():.6g} | max {s.max():.6g}"
                       if non_missing else "all values missing")
        else:
            kind = "text / category"
            if non_missing == 0:
                summary = "all values missing"
            elif distinct == non_missing:
                summary = "unique in every non-missing row"
            else:
                counts = s.value_counts(dropna=True)
                summary = f"mode = {str(counts.index[0])} ({counts.iloc[0] / non_missing:.1%})"

        rows.append({
            "Column": col, "Kind": kind, "Missing": n_missing,
            "Missing (%)": round(100 * n_missing / len(s), 2) if len(s) else np.nan,
            "Distinct": distinct, "Summary": summary
        })

    return pd.DataFrame(rows).set_index("Column")


# These columns are numeric in storage but represent labels/identifiers.
MASTER_CATEGORICAL = {"year", "iso"}
WEEKLY_CATEGORICAL = {"year", "week", "iso"}
master_profile = profile_table(
    master, categorical_columns=MASTER_CATEGORICAL, timestamp_columns={"timestamp"}
)
print(f"Event master: {master.shape[0]:,} rows x {master.shape[1]} columns")
display(master_profile)


**Event-master profile.** The 43 fields include source information and Phase 2 features. Actor/association blanks and the unrecorded targeting label have different meanings; the four constant geographic fields add no variation within this Gaza extract.


In [ ]:
print("Gaza-wide weekly table")
display(profile_table(gaza_weekly, categorical_columns=WEEKLY_CATEGORICAL))


**Gaza-week profile.** Each row is a post-cutoff Gaza week. Early missing lag values mean eligible preceding history is unavailable; the final partial week is excluded from history, rather than being treated as a complete low-activity week.


In [ ]:
print("\nDistrict-week table")
display(profile_table(district_weekly, categorical_columns=WEEKLY_CATEGORICAL))


**District-week profile.** Rows cover five districts over the same calendar. Zero political counts make focal share undefined, while ineligible history creates additional missing lag values; neither should be silently replaced with an observed zero.


In [ ]:
spatial_fields = ["latitude", "longitude", "latitude_rad", "longitude_rad", "geo_precision",
                  "geo_precision_1", "time_precision", "location", "admin2", "coastal_location"]
print("\nLand focal incidents (selected spatial fields only; not all 45 columns)")
display(profile_table(incidents[spatial_fields], categorical_columns={"location", "admin2"}))


**Spatial-input profile.** These selected fields describe the post-cutoff land focal subset, not all event-master records. Coordinates and precision are available, but repeated coordinates and quality categories limit how precisely a point can be interpreted.


In [ ]:
# Free-text and identifier columns: describe length and cardinality rather than a "distribution".
tags_split = master.tags.dropna().str.split(";").explode().str.strip()
text_summary = pd.DataFrame({
    "notes (characters)": master.notes.str.len().describe(percentiles=[.5, .95]).round(0),
}).T
display(text_summary)


**Narrative length.** Event notes vary in length, so a single short excerpt may omit context. This is a description of the text available for review, not a severity score or a direct model feature.


In [ ]:
print("Distinct source strings:", master.source.nunique(), "| distinct locations:", master.location.nunique())
print("\nsource_scale (top 6):")
display(master.source_scale.value_counts().head(6).rename("records").to_frame())


**Source-scale frequencies.** These counts describe the recorded reporting-scale labels; they do not measure source accuracy or establish complete coverage. A combined label remains as recorded.


In [ ]:
print("\nIndividual tag labels (a record can carry several; tags is blank for", master.tags.isna().sum(), "records):")
display(tags_split.value_counts().head(8).rename("records").to_frame())


**Individual tag frequencies.** A record may have several tags, so the tag counts can overlap and should not be added as if they were mutually exclusive events. Blank tags remain unknown or unrecorded.


In [ ]:
print("\nrecord timestamp (Unix seconds, converted):",
      pd.to_datetime(master.timestamp, unit="s").min().date(), "to",
      pd.to_datetime(master.timestamp, unit="s").max().date())


## 1.2 Reported fatalities - common values and rare extremes

**Source:** `acled_events_clean.csv`, all 26,664 records. Fatalities are reported totals for the whole event, not confirmed civilian deaths. The table includes percentiles: P90, for example, is the value at or below which 90% of records fall.

**Figure M1.** Left: small counts, with 11 or more grouped together; middle: cumulative share at or below a fatality count (an ECDF), with a symmetric-log axis that can include zero; right: the inherited `fatalities_log1p = ln(1 + fatalities)` field, with a separate log-scaled vertical count axis. The review line comes from Phase 2's full-export threshold of 23, not Gaza's own P99 of 39.


In [ ]:
f = master.fatalities
REVIEW_THRESHOLD = int(master.loc[~master.high_fatalities_review, "fatalities"].max())
assert master.loc[master.high_fatalities_review, "fatalities"].min() > REVIEW_THRESHOLD

pct_labels = [.5, .75, .9, .95, .99]
fat_summary = pd.Series({
    "Records": len(f),
    "Total reported fatalities": f.sum(),
    "Mean": f.mean(),
    "Median": f.median(),
    "Q1 / Q3": f"{f.quantile(.25):.0f} / {f.quantile(.75):.0f}",
    "P90": f.quantile(.90),
    "P95": f.quantile(.95),
    "P99 (Gaza)": f.quantile(.99),
    "Maximum": f.max(),
    "Records with 0 fatalities": f"{(f == 0).sum():,} ({(f == 0).mean():.1%})",
    "Median among records with 1+": f[f > 0].median(),
    "Mean among records with 1+": round(f[f > 0].mean(), 2),
}).to_frame("fatalities")
display(fat_summary)


**Raw fatality summary.** The median is zero and the upper percentiles extend far above it, showing many zero records and a long right tail. Reported totals refer to the whole event, so neither the mean nor the sum is a count of confirmed civilian deaths.


In [ ]:

log_summary = master.fatalities_log1p.describe(percentiles=[.5, .9, .99]).round(3).to_frame("fatalities_log1p")
display(log_summary.T)


**Log-fatality summary.** This is the inherited natural log of one plus reported fatalities. It compresses large counts and preserves zeros; its values are transformed units, not additional fatalities.


In [ ]:

flagged = master[master.high_fatalities_review]
gaza_p99 = f.quantile(.99)
review = pd.Series({
    "Review threshold (full-export P99, from Phase 2)": f"> {REVIEW_THRESHOLD}",
    "Flagged records": f"{len(flagged):,} ({len(flagged) / len(master):.1%})",
    "Share of all reported fatalities in flagged records": f"{flagged.fatalities.sum() / f.sum():.1%}",
    "Flagged records before Oct 7, 2023": int((flagged.period == BEFORE).sum()),
    "Records above the Gaza-only P99 (comparison)": f"{(f > gaza_p99).sum():,} (P99 = {gaza_p99:.0f})",
}).to_frame("value")
display(review)


**Review-threshold summary.** The Phase 2 flag uses the full-export cutoff greater than 23 and retains 576 Gaza records for review. The Gaza-only 99th percentile is different, so changing thresholds changes which records are flagged rather than proving either group invalid.


In [ ]:
# Individual high-fatality records are reviewed in Task 5 (Figure P2).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# exact counts for small values, with a pooled 11+ bar
vc = f.clip(upper=11).value_counts().sort_index()
axes[0].bar([str(i) if i < 11 else "11+" for i in vc.index], vc.values, color=BLUE)
for x, v in enumerate(vc.values):
    axes[0].text(x, v, f"{v / len(f):.1%}", ha="center", va="bottom", fontsize=8)
axes[0].set_title("A. How many reported fatalities does a typical record carry?", fontsize=10.5)
axes[0].set_xlabel("Reported fatalities per record")
axes[0].set_ylabel("Number of records")

# ECDF on a symmetric-log x-axis
x = np.sort(f.to_numpy())
y = np.arange(1, len(x) + 1) / len(x)
axes[1].step(x, y, where="post", color=GREEN, lw=2)
axes[1].set_xscale("symlog", linthresh=1)
for q, ls in zip([.90, .95, .99], [":", "--", "-."]):
    axes[1].axvline(f.quantile(q), color=GREY, ls=ls, lw=1.2, label=f"P{int(q * 100)} = {f.quantile(q):.0f}")
axes[1].axvline(REVIEW_THRESHOLD, color=RED, lw=1.5, label=f"Review threshold = {REVIEW_THRESHOLD}")
axes[1].set_title("B. How heavy is the tail? (ECDF, log-scaled x)", fontsize=10.5)
axes[1].set_xlabel("Reported fatalities per record (symlog scale)")
axes[1].set_ylabel("Cumulative share of records")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].legend(fontsize=8, loc="lower right")

axes[2].hist(master.fatalities_log1p, bins=30, color=GREEN, edgecolor="white")
axes[2].set_yscale("log")
axes[2].set_title("C. What does the log1p view show? (log-scaled counts)", fontsize=10.5)
axes[2].set_xlabel("log(1 + reported fatalities)")
axes[2].set_ylabel("Number of records (log scale)")

fig.suptitle("Figure M1. Distribution of reported fatalities per Gaza event record (n = 26,664)", fontsize=12.5, y=1.03)
plt.tight_layout()
plt.show()


**Interpretation.** Two thirds of records report zero fatalities (66.1%), while 576 records above the review threshold account for 37.6% of the reported total, so the mean alone is a poor description of a typical record. Medians, percentiles and the log view make the distribution easier to compare, but high values require source review and are not removed.


## 1.3 Event and sub-event types

**Source:** `acled_events_clean.csv`, all records. **Figure M2** shows the category counts and their shares of all 26,664 events; a sub-event type is a more detailed label within an event type. This is the composition of the whole dataset, not the focal share within each type.


In [ ]:
def count_table(series, total=None):
    total = total or len(series)
    out = series.value_counts(dropna=False).rename("Records").to_frame()
    out["Share (%)"] = (100 * out.Records / total).round(2)
    return out

print("event_type")
display(count_table(master.event_type))


**Event-type frequencies.** Explosions/Remote violence dominates the all-event master. These percentages use all records; they differ from the focal prevalence within an event type tested later.


In [ ]:
print("\nsub_event_type (with its parent event_type)")
parent = master.groupby("sub_event_type").event_type.agg(["nunique", "first"])
assert (parent["nunique"] == 1).all(), "A sub_event_type maps to more than one event_type."
sub_tbl = count_table(master.sub_event_type)
sub_tbl.insert(0, "Parent event_type", parent["first"].reindex(sub_tbl.index))
display(sub_tbl)


**Sub-event frequencies.** Air/drone strikes and shelling are the largest detailed categories, while many others have few records. Each sub-event type belongs to one parent type, so these are related descriptions, not independent sets of events.


In [ ]:

print("\ndisorder_type (multi-label values kept as recorded)")
display(count_table(master.disorder_type))


**Disorder-label frequencies.** The table retains combined disorder labels. Political-violence membership uses an exact token check, so a record carrying both Political violence and Demonstrations still belongs in the political denominator.


In [ ]:
both = master.disorder_type.str.contains(";", na=False).sum()
print(f"Records carrying two disorder labels: {both}")
print(f"political_violence flag (token match): {master.political_violence.sum():,} "
      f"({master.political_violence.mean():.1%} of records)")


In [ ]:
def short_label(label, n=46):
    label = str(label)
    return label if len(label) <= n else label[: n - 1] + "…"

def barh_counts(ax, counts, total, title, color=BLUE, xlabel="Number of records"):
    """Horizontal count bars (largest on top) labelled with count and share of `total`."""
    counts = counts.sort_values()
    ax.barh([short_label(i) for i in counts.index], counts.values, color=color)
    for ypos, v in enumerate(counts.values):
        ax.text(v, ypos, f" {v:,} ({v / total:.1%})", va="center", fontsize=8.5)
    ax.set_title(title, fontsize=10.5)
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, counts.max() * 1.30)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.15]})
barh_counts(axes[0], master.event_type.value_counts(), len(master),
            "A. Which event types make up the Gaza records?")
type_order = master.event_type.value_counts().index.tolist()
palette = dict(zip(type_order, sns.color_palette("viridis", len(type_order))))
sub_counts = master.sub_event_type.value_counts().sort_values()
axes[1].barh([short_label(i) for i in sub_counts.index], sub_counts.values,
             color=[palette[parent["first"][i]] for i in sub_counts.index])
for ypos, v in enumerate(sub_counts.values):
    axes[1].text(v, ypos, f" {v:,} ({v / len(master):.1%})", va="center", fontsize=8.5)
axes[1].set_xlim(0, sub_counts.max() * 1.30)
axes[1].set_title("B. Which sub-event types dominate? (colour = parent event type)", fontsize=10.5)
axes[1].set_xlabel("Number of records")
handles = [plt.Rectangle((0, 0), 1, 1, color=palette[t]) for t in type_order]
axes[1].legend(handles, type_order, fontsize=8, loc="lower right", title="Parent event_type")
fig.suptitle("Figure M2. Recorded event and sub-event types (share = % of all 26,664 records)", fontsize=12.5, y=1.01)
plt.tight_layout()
plt.show()


**Interpretation.** Explosions/Remote violence accounts for 76.0% of all records, and air/drone strikes plus shelling/artillery/missile attacks together account for 74.9%. Several categories are very small, so later comparisons show their sample sizes and the hypothesis test uses simulation to handle sparse counts.


## 1.4 Recorded actor labels

**Source:** `acled_events_clean.csv`, with a separate focal-only subset for panel B. **Figure A1** compares actor labels and actor types; blank actor2 values are shown explicitly because the corresponding records describe single-sided interactions. Actor1 and actor2 are recorded positions in the coding, not verified legal or operational roles.


In [ ]:
focal = master[master.focal_event]

actor_overview = pd.DataFrame({
    "actor1": [master.actor1.isna().sum(), master.actor1.nunique()],
    "actor2": [master.actor2.isna().sum(), master.actor2.nunique()],
    "inter1 (actor type)": [master.inter1.isna().sum(), master.inter1.nunique()],
    "inter2 (actor type)": [master.inter2.isna().sum(), master.inter2.nunique()],
}, index=["Missing", "Distinct labels"]).T
actor_overview["Missing (%)"] = (100 * actor_overview.Missing / len(master)).round(2)
display(actor_overview)


**Actor-field completeness.** First-actor fields are complete, while second-actor and second-type blanks occur together. The check below confirms these blanks belong to single-sided interaction coding rather than providing evidence of an unidentified second actor.


In [ ]:

only = master.interaction.str.endswith(" only", na=False)
assert master.actor2.isna().equals(master.inter2.isna()) and (~master.actor2.isna() | only).all()
print("Blank actor2 = blank inter2 in every record, and every blank sits on a single-sided ('only') interaction.")

print("\nTop actor1 labels, all events")
display(count_table(master.actor1).head(10))


**First actors across all events.** A small number of labels account for most records. These are original ACLED labels across both periods and all event types, not verified assignments of responsibility.


In [ ]:
print("\nTop actor2 labels, all events (blank shown as NaN)")
display(count_table(master.actor2).head(8))


**Second actors across all events.** Civilians (Palestine) is frequent, and the missing category represents records without a coded second actor. A blank should not be interpreted as a named participant or as evidence that civilians were unharmed.


In [ ]:
print(f"\nFocal events: {len(focal):,}")
print("Top actor1 labels among focal events")
display(count_table(focal.actor1).head(8))


**First actors within focal events.** Military Forces of Israel (2022-) accounts for 97.9% of this subset. The denominator is focal events, so these shares should not be confused with the preceding all-event actor frequencies.


In [ ]:
print("actor2 among focal events:", focal.actor2.value_counts(dropna=False).to_dict())

print("\nActor-type (inter1) counts, all events")
display(count_table(master.inter1))


**First-actor types.** This broader category table describes the types assigned to actor1, rather than distinct individual actors. Category prevalence alone does not verify roles in a particular incident.


In [ ]:

# Date range of the two Israeli-forces labels: are they versions rather than separate actors?
israel = master[master.actor1.str.startswith("Military Forces of Israel", na=False)]
display(israel.groupby("actor1").event_date.agg(["min", "max", "size"]))


**Date-versioned Israeli-forces labels.** The date ranges show the older label ending before the newer label begins. Both source labels remain separate; interpreting or merging them requires a documented analyst decision.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))

barh_counts(axes[0, 0], master.actor1.value_counts().head(10), len(master),
            "A. Which actor1 labels recur across all events?")
barh_counts(axes[0, 1], focal.actor1.value_counts().head(8), len(focal),
            "B. Which actor1 labels recur among focal events? (n = 10,714)", color=RED)

actor2_counts = master.actor2.fillna("(blank: single-sided event)").value_counts().head(8)
barh_counts(axes[1, 0], actor2_counts, len(master),
            "C. Which actor2 labels recur across all events? (blank shown)", color=GREEN)
barh_counts(axes[1, 1], master.inter1.value_counts(), len(master),
            "D. Which actor types (inter1) are recorded as the first actor?", color=GREY)

fig.suptitle("Figure A1. Recorded actor labels (labels are ACLED coding, not verified roles)", fontsize=12.5, y=1.0)
plt.tight_layout()
save_supporting_figure("A1", fig)


The supporting figure and its interpretation are in [Appendix A1](#A1---Recorded-actor-profiles); the calculations and tables above remain available here.


In [ ]:
col = 'targeting_flagged'
counts = master[col].value_counts(dropna=False).reindex([True, False], fill_value=0)
table = pd.DataFrame({"count": counts, "share_pct": (100 * counts / len(master)).round(2)})
print(f"{col}: denominator = all {len(master):,} Gaza event records")
display(table)


**Targeting-flag counts.** True means ACLED recorded the Civilian targeting label; False means that label was not recorded. This is broader than the project's focal rule and is not a complete census of civilian harm.


In [ ]:
col = 'political_violence'
counts = master[col].value_counts(dropna=False).reindex([True, False], fill_value=0)
table = pd.DataFrame({"count": counts, "share_pct": (100 * counts / len(master)).round(2)})
print(f"{col}: denominator = all {len(master):,} Gaza event records")
display(table)


**Political-violence counts.** This flag identifies the comparison population used for focal shares. Its denominator here is all Gaza master records, whereas later focal-share calculations divide only by political records.


In [ ]:
col = 'focal_event'
counts = master[col].value_counts(dropna=False).reindex([True, False], fill_value=0)
table = pd.DataFrame({"count": counts, "share_pct": (100 * counts / len(master)).round(2)})
print(f"{col}: denominator = all {len(master):,} Gaza event records")
display(table)


**Focal-rule counts.** True requires both the targeting flag and the exact Palestinian civilian actor label. False means the rule is not met; within political-only comparisons it denotes the other political records, not proof of no civilian harm.


**Reading the flag tables.** These are shares of **all event records**: `targeting_flagged` indicates the recorded source flag, `political_violence` selects the later comparison group, and `focal_event` adds the exact civilian-actor rule. They have different meanings and should not be used interchangeably.


## 1.5 Location and date precision

**Sources:** the event master and land focal-incident file. **Figure M3** compares the proportions in each recorded precision category; code 1 is the most precise category. The spatial file is already a post-cutoff focal subset, so differences from the all-period master reflect both cohort selection and recording precision.


In [ ]:
def code_share(frame, col):
    return (frame[col].value_counts(normalize=True).sort_index() * 100).round(2)

prec = pd.DataFrame({
    "geo_precision: master (n)": master.geo_precision.value_counts().sort_index(),
    "geo_precision: master (%)": code_share(master, "geo_precision"),
    "geo_precision: land focal incidents (n)": incidents.geo_precision.value_counts().sort_index(),
    "geo_precision: land focal incidents (%)": code_share(incidents, "geo_precision"),
})
display(prec)


**Location-precision comparison.** Precision 1 is more prevalent in the land focal subset than in the full master. The populations also differ by period and focal/land restrictions, so this difference cannot be attributed to precision alone.


In [ ]:
tprec = pd.DataFrame({
    "time_precision: master (n)": master.time_precision.value_counts().sort_index(),
    "time_precision: master (%)": code_share(master, "time_precision"),
    "time_precision: land focal incidents (n)": incidents.time_precision.value_counts().sort_index(),
    "time_precision: land focal incidents (%)": code_share(incidents, "time_precision"),
})
display(tprec)


**Date-precision comparison.** Code 1 predominates in both tables, but the less precise records remain identifiable. Their small proportion does not itself prove that shifting uncertain dates would leave weekly results unchanged.


In [ ]:
by_period_prec = pd.crosstab(master.period, master.geo_precision, normalize="index").mul(100).round(1)
by_period_prec.columns = [f"geo_precision {i} (%)" for i in by_period_prec.columns]
print("geo_precision by period (share of each period's records):")
display(by_period_prec.loc[[BEFORE, AFTER]])


**Location precision by period.** The proportion at precision 1 is substantially higher after the cutoff. Before/after spatial comparisons therefore have different recording-quality mixes, motivating the later within-period precision checks.


In [ ]:
print("geo_precision_1 =", f"{master.geo_precision_1.mean():.1%} of master;", f"{incidents.geo_precision_1.mean():.1%} of land focal incidents")
print("time_precision_1 =", f"{master.time_precision_1.mean():.1%} of master")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, col, title in [(axes[0], "geo_precision", "A. How precisely are locations recorded?"),
                       (axes[1], "time_precision", "B. How precisely are dates recorded?")]:
    shares = pd.DataFrame({"Event master (n = 26,664)": master[col].value_counts(normalize=True).sort_index() * 100,
                           "Land focal incidents (n = 10,583)": incidents[col].value_counts(normalize=True).sort_index() * 100})
    shares.index = [f"Code {i}" for i in shares.index]
    shares.plot.bar(ax=ax, color=[BLUE, RED], rot=0, width=0.75)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", fontsize=8.5)
    ax.set_title(title, fontsize=10.5)
    ax.set_ylabel("% of records in the table")
    ax.set_xlabel("Precision code (category, not a measurement)")
    ax.set_ylim(0, 112)
    ax.legend(fontsize=8.5)
fig.suptitle("Figure M3. Recorded precision codes", fontsize=12.5, y=1.02)
plt.tight_layout()
plt.show()


**Interpretation.** Date precision is code 1 for 99.2% of master records, whereas location precision is code 1 for only 53.7%; the land focal subset retains a higher 71.6% at location precision 1. Restricting to that subset removes about 28% of spatial incidents and changes which places contribute records, so both full and restricted spatial views are needed.


## 1.6 Where and when all events are recorded

**Source:** `acled_events_clean.csv`, both periods and all event types. **Figure A2** shows district counts, retaining a missing-district category, and totals by event-calendar month. These totals are neither post-cutoff focal counts nor time-adjusted rates.


In [ ]:
admin2_counts = master.admin2.fillna("(missing district)").value_counts()
admin_tbl = admin2_counts.rename("Records").to_frame()
admin_tbl["Share of all events (%)"] = (100 * admin_tbl.Records / len(master)).round(2)
admin_tbl["Share of known-district events (%)"] = (100 * admin_tbl.Records / master.admin2.notna().sum()).round(2)
admin_tbl.loc["(missing district)", "Share of known-district events (%)"] = np.nan
display(admin_tbl)


**District frequencies.** Gaza City has the most all-event records, while four records have no known district. Their share among known districts is left undefined; they remain in Gaza-wide counts rather than being assigned to a district.


In [ ]:
print("Missing-district records (all coast-coded in Phase 2):", int(master.district_missing.sum()))
print(f"Coast-coded locations: {master.coastal_location.sum():,} records ({master.coastal_location.mean():.1%}); "
      f"distinct location labels: {master.location.nunique()}")

display(master.groupby("year").size().rename("Records").to_frame().assign(
    **{"Share (%)": lambda d: (100 * d.Records / len(master)).round(2)}))


**Annual frequencies.** These are counts of all recorded events, with 2025 covering January-September only. Different coverage lengths mean raw annual totals are not directly comparable annual rates.


In [ ]:

monthly = master.groupby("month_start").size()
weekly_n = master.groupby("week_start").size()
print(f"\nDistinct calendar months: {monthly.size} | distinct Saturday-start weeks: {weekly_n.size}")
display(pd.DataFrame({"Records per month": monthly.describe().round(1),
                      "Records per week": weekly_n.describe().round(1)}).T)


**Monthly and weekly count summaries.** These distributions use all-event calendar-month and Saturday-start-week totals from the master, including boundary periods. They summarize recording volume, not post-cutoff focal counts or equal-duration exposure.


In [ ]:
print("Peak month:", monthly.idxmax().strftime("%Y-%m"), f"({monthly.max():,} records)")
print("Peak week:", weekly_n.idxmax().date(), f"({weekly_n.max():,} records)")
print("Median monthly records, before vs from Oct 2023:",
      int(monthly[monthly.index < "2023-10-01"].median()), "vs", int(monthly[monthly.index >= "2023-10-01"].median()))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 1.6]})
barh_counts(axes[0], admin2_counts, len(master), "A. How are records spread across districts (admin2)?")
axes[1].bar(monthly.index, monthly.values, width=24, color=BLUE)
axes[1].axvline(cutoff, color=RED, ls="--", lw=1.5, label="Oct 7, 2023")
axes[1].set_title("B. How many event records are logged each month?", fontsize=10.5)
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Number of records")
axes[1].legend(fontsize=9)
fig.suptitle("Figure A2. Records by district and by month (event master)", fontsize=12.5, y=1.02)
plt.tight_layout()
save_supporting_figure("A2", fig)


The supporting figure and its interpretation are in [Appendix A2](#A2---All-event-district-and-month-counts); the calculations and tables above remain available here.


# Task 2 - How do groups differ?

The first task described all records; this task asks whether the focal proportion, reported severity and timing differ between groups. We now use political-violence records for focal-share comparisons, and switch explicitly to weekly CSVs for temporal analyses.


**Source guide:** event comparisons use `master`, filtered to `political_violence` unless marked focal-only; district comparisons use `district_week`; Gaza-wide series use `gaza_week`. Counts are never added across event and weekly tables because those are two views of the same events.

The event-type prevalence comparison is presented once, with its statistical intervals and test, in [Task 4, Figure M9](#4.3-Hypothesis-B---Does-focal-share-differ-by-event-type?).


## 2.1 Before and from October 7: counts, shares and rates

**Source:** political records from the event master. The table compares counts, focal shares and rates adjusted for the 644/725 observation days; the existing rate plot is combined with Hypothesis A in [Figure M8](#Figure-M8---Period-rates,-focal-shares-and-within-type-evidence).


In [ ]:
t1 = (political.groupby("period")
      .agg(political_records=("event_id_cnty", "size"), focal=("focal_event", "sum"))
      .loc[[BEFORE, AFTER]])
t1["not_focal"] = t1.political_records - t1.focal
t1["focal_share"] = t1.focal / t1.political_records
t1["calendar_days"] = [PERIOD_DAYS[p] for p in t1.index]
t1["focal_per_week"] = t1.focal * 7 / t1.calendar_days
t1["political_per_week"] = t1.political_records * 7 / t1.calendar_days
display(t1.round({"focal_share": 3, "focal_per_week": 2, "political_per_week": 2}))


def draw_period_rates(fig, ax):
    x = np.arange(len(t1)); w = 0.36
    b1 = ax.bar(x - w / 2, t1.political_per_week, w, color=C_OTHER, label="Political-violence records")
    b2 = ax.bar(x + w / 2, t1.focal_per_week, w, color=C_FOCAL, label="Focal events")
    for bars_, vals in ((b1, t1.political_per_week), (b2, t1.focal_per_week)):
        for b, v in zip(bars_, vals):
            ax.text(b.get_x() + b.get_width() / 2, v * 1.12, f"{v:,.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_yscale("log"); ax.set_ylim(0.3, 700)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{p}\n({PERIOD_DAYS[p]} days)" for p in t1.index])
    ax.set_xlabel("Period (calendar days)")
    ax.set_ylabel("Records per week (log scale)")
    ax.set_title("How many records per week does each period\nhold, once period length is accounted for?")
    ax.legend(loc="upper left")


The period table shows a rise from 16.2% to 47.6% focal share and from 0.68 to 102.8 focal events per seven days. [Figure M8](#Figure-M8---Period-rates,-focal-shares-and-within-type-evidence) puts the rate and interval comparisons together, so this comparison is drawn once.


## 2.2 Reported fatalities by focal status

**Source:** political-violence records from the event master, including zeros. **Figure A3** uses an ECDF: at any fatality count on the horizontal axis, the line shows the share of that group at or below it. Tables also split the comparison by period.


In [ ]:
def fatality_summary(df, by):
    g = df.groupby(by, observed=True).fatalities
    return pd.DataFrame({"n": g.size(), "share_zero": g.apply(lambda s: (s == 0).mean()),
                         "median": g.median(), "q1": g.quantile(0.25), "q3": g.quantile(0.75),
                         "p90": g.quantile(0.90), "mean": g.mean(), "max": g.max()})

print("Political-violence records, by focal status")
display(fatality_summary(political, "focal_event").round(2))
print("Same comparison split by period")
display(fatality_summary(political, ["period", "focal_event"]).round(2))

fig, ax = plt.subplots(figsize=(9.5, 5))
for flag, label, col in [(True, "Focal", C_FOCAL), (False, "Not focal", C_OTHER)]:
    v = np.sort(political.loc[political.focal_event == flag, "fatalities"].to_numpy())
    ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", color=col, lw=2, label=f"{label} (n = {len(v):,})")
ax.set_xscale("symlog", linthresh=1)
ticks = [0, 1, 2, 5, 10, 25, 50, 100, 250, 500]
ax.set_xticks(ticks); ax.set_xticklabels([str(t) for t in ticks]); ax.minorticks_off()
ax.set_xlim(0, 600); ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Reported fatalities per event (symmetric-log scale)")
ax.set_ylabel("Share of records at or below this value (ECDF)")
ax.set_title("Do focal records report more fatalities than other\npolitical-violence records?")
ax.legend(loc="lower right")
plt.tight_layout()
save_supporting_figure("A3", fig)


The supporting figure and its interpretation are in [Appendix A3](#A3---Fatality-ECDF-by-focal-status); the calculations and tables above remain available here.


## 2.3 How often is a fatality recorded, and how large is the total?

**Source:** event-master political records grouped by event type and focal status. **Figure M4** separates two questions: left, the share reporting at least one fatality; right, the distribution **only among positive-fatality records**. Groups with fewer than 30 records remain in the table but are omitted from the figure; box centers show medians and boxes the middle half of values.


In [ ]:
short = {"Explosions/Remote violence": "Explosions", "Violence against civilians": "Violence vs. civilians"}
political["group"] = (political.event_type.map(lambda t: short.get(t, t))
                      + np.where(political.focal_event, " · focal", " · not focal"))

g5 = political.groupby("group").fatalities
t5 = pd.DataFrame({"n": g5.size(), "share_with_fatality": g5.apply(lambda s: (s > 0).mean()),
                   "events_with_fatality": g5.apply(lambda s: (s > 0).sum()),
                   "median_if_any": g5.apply(lambda s: s[s > 0].median()),
                   "q1_if_any": g5.apply(lambda s: s[s > 0].quantile(0.25)),
                   "q3_if_any": g5.apply(lambda s: s[s > 0].quantile(0.75)),
                   "max": g5.max()}).sort_values("n", ascending=False)
t5["shown_in_chart"] = t5.n >= 30
display(t5.round(3))

order = [g for g in t5.index[t5.n >= 30]]
palette5 = {g: (C_FOCAL if g.endswith("· focal") else C_OTHER) for g in order}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

ax = axes[0]
vals = t5.loc[order, "share_with_fatality"]
bars = ax.barh(order, vals, color=[palette5[g] for g in order])
for y, g in enumerate(order):
    r = t5.loc[g]
    ax.text(r.share_with_fatality + 0.01, y, f"{r.events_with_fatality:,.0f} / {r.n:,.0f} = {r.share_with_fatality:.0%}", va="center", fontsize=9)
ax.invert_yaxis(); ax.set_xlim(0, 1.0); ax.grid(axis="x")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Records with at least 1 reported fatality / all records in group")
ax.set_ylabel("Event type · focal status")
ax.set_title("How often does each group report any fatality?")

ax = axes[1]
sub = political[political.group.isin(order) & (political.fatalities > 0)]
sns.boxplot(data=sub, x="fatalities", y="group", hue="group", order=order, hue_order=order,
            palette=palette5, legend=False, orient="h", fliersize=2, linewidth=1, ax=ax)
ax.set_xscale("log")
ax.set_yticks(range(len(order)))
ax.set_yticklabels([f"{g}\n(n = {int((sub.group == g).sum()):,})" for g in order], fontsize=9)
ax.set_xlabel("Reported fatalities per event (log scale), records with at least 1")
ax.set_ylabel("")
ax.set_title("Among records reporting a fatality, how large\nare the reported totals?")
plt.tight_layout()
plt.show()


**Interpretation.** Within Explosions, 69% of focal records report a fatality versus 8% of non-focal records, yet both positive-fatality groups have median 3. The main difference is therefore how often a fatality is recorded, rather than a higher conditional median; all totals still refer to entire event records.


## 2.4 Does location precision change the focal share?

**Source:** event-master political records, grouped by `period` and `geo_precision`. **Figure M5** keeps each group's own political-record denominator; the supporting tables check the later period within districts and within Explosions. Precision is a recording characteristic, so this is a sensitivity comparison, not evidence that precision causes targeting.


In [ ]:
t6 = (political.groupby(["period", "geo_precision"], observed=True)
      .agg(n=("event_id_cnty", "size"), focal=("focal_event", "sum")))
t6["focal_share"] = t6.focal / t6.n
display(t6.round({"focal_share": 3}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, period, col in zip(axes, [BEFORE, AFTER], [C_GREY, C_FOCAL]):
    d = t6.loc[period].reindex([1, 2, 3])
    bars = ax.bar([f"Precision {k}" for k in d.index], d.focal_share, color=col, width=0.6)
    for b, (_, r) in zip(bars, d.iterrows()):
        if r.n < 30:
            b.set_hatch("//"); b.set_alpha(0.55)
        ax.text(b.get_x() + b.get_width() / 2, r.focal_share + 0.015,
                f"{r.focal:,.0f} / {r.n:,.0f}\n{r.focal_share:.1%}", ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, 0.85)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Location precision code (1 = most precise)")
    ax.set_title(f"{period}\n(n = {int(d.n.sum()):,} political-violence records)")
axes[0].set_ylabel("Focal events / political-violence records")
fig.suptitle("Is the focal share different for records with less precise locations, within each period?",
             fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Robustness: does the precision 1 vs 2 gap survive inside each district and inside Explosions? (Oct 7 onward)
post_pol = political[political.period == AFTER]
by_district = post_pol[post_pol.geo_precision.isin([1, 2])].groupby(["admin2", "geo_precision"], observed=True).focal_event.agg(["size", "mean"]).unstack()
by_district.columns = [f"{'n' if a == 'size' else 'focal share'}, precision {b}" for a, b in by_district.columns]
print("Oct 7 onward: focal share by district and precision (precision 3 omitted, n < 70 in every district)")
display(by_district.round(3))
expl = post_pol[post_pol.event_type == "Explosions/Remote violence"]
print("Oct 7 onward, Explosions/Remote violence only")
display(expl.groupby("geo_precision", observed=True).focal_event.agg(n="size", focal="sum", focal_share="mean").round(3))


**Interpretation.** From the cutoff onward, focal share is 59.6% at precision 1 versus 31.7% at precision 2, with the same ordering before the cutoff (21.7% versus 14.2%). The later-period gap also appears within districts and within Explosions, supporting a precision sensitivity check while leaving reporting and coding explanations unresolved.


## 2.5 Which actor labels and sub-event types occur together?

**Source:** all-period focal records from `acled_events_clean.csv`. **Figure M6** is a cross-tab heatmap: each cell counts one actor1/sub-event combination, the five most frequent actors have their own rows, and the rest share a display-only row. Color uses a log scale so smaller combinations remain visible; blank cells mean zero recorded events.


In [ ]:
focal = master.loc[master.focal_event].copy()
print("Focal events:", len(focal))
print("actor2 == 'Civilians (Palestine)':", int((focal.actor2 == "Civilians (Palestine)").sum()),
      "| actor2 blank:", int(focal.actor2.isna().sum()),
      "| actor1 == 'Civilians (Palestine)':", int((focal.actor1 == "Civilians (Palestine)").sum()))
print("Distinct actor1 labels among focal events:", focal.actor1.nunique())
print("\nEvent type vs. sub-event type among focal events")
display(pd.crosstab(focal.sub_event_type, focal.event_type))

ct = pd.crosstab(focal.actor1, focal.sub_event_type)
totals = ct.sum(axis=1).sort_values(ascending=False)
TOP_N = 5
keep = totals.index[:TOP_N]
ct_show = ct.loc[keep].copy()
ct_show.loc[f"All other actor1 labels ({len(totals) - TOP_N} labels)"] = ct.loc[~ct.index.isin(keep)].sum()
ct_show = ct_show[ct_show.sum().sort_values(ascending=False).index]
ct_show.index = [f"{i}  (n = {n:,})" for i, n in ct_show.sum(axis=1).items()]
display(ct_show)

fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(ct_show.where(ct_show > 0), annot=ct_show.astype(int).astype(str), fmt="",
            cmap="Reds", norm=LogNorm(vmin=1, vmax=ct_show.to_numpy().max()),
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Focal events (log colour scale)"}, ax=ax)
ax.grid(False)
ax.set_xlabel("Sub-event type (blank cell = 0 events)")
ax.set_ylabel("Recorded actor1 label")
ax.set_title("Which recorded actor1 labels appear in focal events, and under which sub-event types?")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.show()


**Interpretation.** Military Forces of Israel (2022-) is the actor1 label in 10,493 of 10,714 focal records, including 8,392 air/drone-strike and 1,436 shelling records for that actor. Across all focal actors the corresponding totals are 8,420 and 1,442, and the coding alone does not verify operational roles or responsibility.


## 2.6 Did the mix of event types change?

**Source:** event-master political records, grouped by period and type. **Figure A4** shows each type's share of all political records within its period, so each bar sums to 100%; the accompanying table also records focal counts within each type. The separate question of whether focal prevalence changed **inside** a type is tested in [Figure M8](#Figure-M8---Period-rates,-focal-shares-and-within-type-evidence).


In [ ]:
type_order = ["Explosions/Remote violence", "Battles", "Violence against civilians", "Protests", "Riots"]
t7 = (political.groupby(["period", "event_type"])
      .agg(records=("event_id_cnty", "size"), focal=("focal_event", "sum")))
t7["focal_share"] = t7.focal / t7.records
display(t7.round({"focal_share": 3}))

comp = (pd.crosstab(political.period, political.event_type)
        .reindex(index=[BEFORE, AFTER], columns=type_order, fill_value=0))
comp_share = comp.div(comp.sum(axis=1), axis=0)
type_colors = dict(zip(type_order, ["#315A7D", "#9AA5B1", "#8D3B42", "#E0A458", "#6B8F71"]))

fig, ax = plt.subplots(figsize=(11, 5))
for i, period in enumerate([BEFORE, AFTER]):
    left = 0
    for t in type_order:
        wdt = comp_share.loc[period, t]
        ax.barh(i, wdt, left=left, color=type_colors[t], edgecolor="white", label=t if i == 0 else None)
        if wdt >= 0.03:
            ax.text(left + wdt / 2, i, f"{wdt:.0%}", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
        left += wdt
ax.set_yticks([0, 1])
ax.set_yticklabels([f"{p}\n(n = {int(comp.loc[p].sum()):,})" for p in [BEFORE, AFTER]])
ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Share of the period's political-violence records")
ax.set_ylabel("Period")
ax.set_title("Did the event-type mix of political-violence\nrecords change between periods?")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)

plt.tight_layout(); save_supporting_figure("A4", fig)


The supporting figure and its interpretation are in [Appendix A4](#A4---Event-type-composition-by-period); the calculations and tables above remain available here.


## 2.7 District-week counts

**Source:** `acled_district_week.csv`. The table includes all 104 weeks, while the existing count heatmap masks the final partial week and is combined with district variability in [Figure M12](#Figure-M12---District-counts,-typical-volume-and-variability). Counts represent recorded volume; [Appendix A5](#A5---Within-district-focal-prevalence) separately examines within-district focal/political shares.


In [ ]:
dw = district_week.copy()
dist_order = dw.groupby("admin2").focal_events.sum().sort_values(ascending=False).index
heat = dw.pivot(index="admin2", columns="week_start", values="focal_events").loc[dist_order].astype(float)
complete = dw.drop_duplicates("week_start").set_index("week_start").complete_week.reindex(heat.columns).to_numpy()
heat_plot = heat.copy()
heat_plot.loc[:, ~complete] = np.nan  # keep the partial week out of the colour scale

dist_summary = dw.groupby("admin2").agg(focal=("focal_events", "sum"), political=("political_events", "sum"),
                                         median_weekly_focal=("focal_events", "median"),
                                         max_weekly_focal=("focal_events", "max"),
                                         weeks_without_political=("political_events", lambda s: int((s == 0).sum()))).loc[dist_order]
dist_summary["focal_share"] = dist_summary.focal / dist_summary.political
display(dist_summary.round({"focal_share": 3}))
print("Highest district-weeks by focal count")
display(dw.nlargest(6, "focal_events")[["admin2", "week_start", "focal_events", "political_events", "focal_share"]].round({"focal_share": 3}))


def draw_district_heatmap(fig, ax):
    sns.heatmap(heat_plot, cmap="rocket_r", xticklabels=False, cbar_kws={"label": "Recorded focal events per week", "pad": 0.015}, ax=ax)
    for j in np.where(~complete)[0]:
        ax.add_patch(Rectangle((j, 0), 1, len(heat), facecolor="#D9D9D9", edgecolor="none", hatch="///"))
        ax.text(j + 0.5, -0.15, "partial week", ha="right", va="bottom", fontsize=8)
    first_of_quarter = ~pd.Series(pd.PeriodIndex(heat.columns, freq="Q")).duplicated().to_numpy()
    pos = np.where(first_of_quarter)[0]
    ax.set_xticks(pos + 0.5)
    ax.set_xticklabels(heat.columns[pos].strftime("%b %Y"), rotation=0, fontsize=9)
    ax.grid(False)
    ax.set_xlabel("Week start (Saturday), labelled at the first week of each quarter")
    ax.set_ylabel("District (admin2)")
    ax.set_title("Which Gaza districts have high focal-event weeks, and when?")


The largest district-week is Gaza City, November 4, 2023, with 102 focal records. Its timing and the shared early-2025 decline can be examined together with district variability in [Figure M12](#Figure-M12---District-counts,-typical-volume-and-variability).


## 2.8 Weekly counts and focal share

**Source:** `acled_gaza_week.csv`, with a pooled post-cutoff share from matching master records. The tables identify the five lowest focal-count weeks and quarterly weekly means; the share panel is combined with the existing anomaly-count timelines in [Figure M10](#5.3-Which-Gaza-wide-weeks-stand-out?). This retains one integrated weekly overview rather than separate repeated count timelines.


In [ ]:
gw = gaza_week.sort_values("week_start").copy()
low5 = gw.nsmallest(5, "focal_events")
low_span = (low5.week_start.min(), low5.week_end.max())
partial = gw[~gw.complete_week]
full = gw[gw.complete_week]
post_share = master.loc[master.period == AFTER, "focal_event"].sum() / political.loc[political.period == AFTER].shape[0]
print("Five lowest focal-count weeks:", [d.date().isoformat() for d in sorted(low5.week_start)])
display(low5.sort_values("week_start")[["week_start", "recorded_events", "political_events", "focal_events", "focal_share"]].round({"focal_share": 3}))
print("Mean weekly focal events by quarter (complete weeks)")
display(full.groupby(full.week_start.dt.to_period("Q"))[["focal_events", "political_events"]].mean().round(1))


def draw_weekly_share(fig, ax):
    ax.plot(full.week_start, full.focal_share, color=C_OTHER, lw=1.8, label="Weekly focal share")
    ax.scatter(partial.week_start, partial.focal_share, facecolor="white", edgecolor=C_OTHER, zorder=5, label="Incomplete final week")
    ax.axhline(post_share, color="black", ls="--", lw=1, label=f"Pooled share, Oct 7 onward ({post_share:.1%})")
    ax.axvspan(*low_span, color="#E0A458", alpha=0.25)
    ax.set_ylim(0, 0.85)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_ylabel("Focal events / political-violence records")
    ax.set_xlabel("Week start (Saturday)")
    ax.set_title("What share of each week's political-violence records is focal?")
    ax.legend(loc="upper right", ncol=3)


The Gaza-wide peak is 254 focal events in the week starting October 28, 2023, and the five lowest focal-count weeks have 15-23 records. [Figure M10](#5.3-Which-Gaza-wide-weeks-stand-out?) shows why those low counts, focal share and unusual-week flags need to be read together.


## 2.9 How does focal share vary within districts?

**Source:** `acled_district_week.csv`, **103 complete weeks per district**. **Figure A5** compares a district's focal events with its own political events: grey is the weekly share, red is the sum of focal events over the current and preceding three weeks divided by the matching political sum, and the dashed line is the complete-period pooled share. This four-week smoothing includes the current week and is not the inherited preceding-four-week history feature; gaps indicate zero political denominators.


In [ ]:
dwc = district_week[district_week.complete_week].sort_values(["admin2", "week_start"]).copy()
roll = lambda col: dwc.groupby("admin2")[col].transform(lambda s: s.rolling(4, min_periods=4).sum())
dwc["focal_4w"], dwc["political_4w"] = roll("focal_events"), roll("political_events")
dwc["share_4w"] = dwc.focal_4w / dwc.political_4w.where(dwc.political_4w > 0)

tbl = (dwc.groupby("admin2").agg(focal=("focal_events", "sum"), political=("political_events", "sum"),
                                  undefined_weeks=("focal_share", lambda s: int(s.isna().sum()))).loc[dist_order])
tbl["overall_share"] = tbl.focal / tbl.political
display(tbl.round({"overall_share": 3}))

fig, axes = plt.subplots(3, 2, figsize=(13, 9.5), sharex=True, sharey=True)
axes = axes.ravel()
for ax, dname in zip(axes, dist_order):
    d = dwc[dwc.admin2 == dname]
    ax.plot(d.week_start, d.focal_share, color=C_GREY, lw=0.9, label="Weekly share")
    ax.plot(d.week_start, d.share_4w, color=C_FOCAL, lw=2, label="4-week pooled share")
    ax.axhline(tbl.loc[dname, "overall_share"], color="black", ls="--", lw=1, label="District overall share")
    ax.set_title(f"{dname}: {tbl.loc[dname, 'focal']:,.0f} focal / {tbl.loc[dname, 'political']:,.0f} political records")
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.tick_params(axis="x", labelrotation=30)
axes[0].legend(loc="lower left", fontsize=8)
axes[3].tick_params(axis="x", labelbottom=True, labelrotation=30)
for ax in axes[-3:-1]:
    ax.set_xlabel("Week start")
for ax in axes[::2][:3]:
    ax.set_ylabel("Focal share")
# Last panel: the denominators as a table
ax = axes[-1]; ax.axis("off")
cell = [[i, f"{r.focal:,.0f}", f"{r.political:,.0f}", f"{r.overall_share:.1%}", int(r.undefined_weeks)] for i, r in tbl.iterrows()]
t = ax.table(cellText=cell, colLabels=["District", "Focal", "Political", "Share", "Weeks w/o\npolitical"], loc="center", cellLoc="center")
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.6)
ax.set_title("Denominators (complete weeks only)")
fig.suptitle("Do districts differ in focal share, and how stable is the share from week to week?", fontsize=12, fontweight="bold", y=1.0)
plt.tight_layout()
save_supporting_figure("A5", fig)


The supporting figure and its interpretation are in [Appendix A5](#A5---Within-district-focal-prevalence); the calculations and tables above remain available here.


## 2.10 Which measures move together, and which categories are associated?

**Figure M7 uses two sources and two different measures.** Left: `acled_gaza_week.csv`, complete weeks, with **Spearman rank correlation** from -1 to +1 (opposite to similar ordering of high/low values). Right: political records from `acled_events_clean.csv`, with **Cramer's V** from 0 to 1 (weaker to stronger categorical association, without a direction).

The two history fields on the left are inherited eligible prior-week counts and the sum of four preceding weeks. Early unavailable history reduces the number of pairs; the pair-count table makes that explicit. A dagger marks built-in count/share or overlapping-window relationships, and a double dagger marks current counts compared with their earlier values.

On the right, records missing either category are omitted for that pair. Sub-event type is nested in event type, so their perfect association is built into the labels; sparse categories can inflate uncorrected V. Neither matrix demonstrates causation.


In [ ]:
# ---- Left: Spearman on weekly Gaza-wide numbers (complete weeks only) ----
wk_cols = ["recorded_events", "political_events", "focal_events", "focal_share", "previous_week_focal", "previous_four_weeks_focal"]
wk_lab = {"recorded_events": "Recorded events", "political_events": "Political-violence records", "focal_events": "Focal events",
          "focal_share": "Focal share", "previous_week_focal": "Focal events, previous week", "previous_four_weeks_focal": "Focal events, previous 4 weeks"}
wk = gaza_week.loc[gaza_week.complete_week, wk_cols]
rho = wk.corr(method="spearman")
n_pair = wk.notna().astype(int).T.dot(wk.notna().astype(int))
mechanical = [{"recorded_events", "political_events"}, {"political_events", "focal_events"},
              {"focal_events", "focal_share"}, {"political_events", "focal_share"},
              {"previous_week_focal", "previous_four_weeks_focal"}]
serial = [{"focal_events", "previous_week_focal"}, {"focal_events", "previous_four_weeks_focal"}]
def mark(a, b):
    s = {a, b}
    return "†" if s in mechanical else ("‡" if s in serial else "")
annot_r = pd.DataFrame([[f"{rho.loc[a, b]:.2f}{mark(a, b)}" if a != b else "1" for b in wk_cols] for a in wk_cols], index=wk_cols, columns=wk_cols)
print(f"Weekly Spearman: {len(wk)} complete weeks; pairwise n ranges from {int(n_pair.to_numpy().min())} to {int(n_pair.to_numpy().max())}")

# ---- Right: Cramér's V (plain, uncorrected) on political-violence records ----
def cramers_v(a, b):
    ok = a.notna() & b.notna()
    ct = pd.crosstab(a[ok].astype(str), b[ok].astype(str))
    if min(ct.shape) < 2:
        return np.nan, np.nan
    chi2, _, _, expected = chi2_contingency(ct, correction=False)
    return np.sqrt(chi2 / (ct.to_numpy().sum() * (min(ct.shape) - 1))), (expected < 5).mean()

cat_vars = {"Focal event": political.focal_event, "Event type": political.event_type, "Sub-event type": political.sub_event_type,
            "Period": political.period, "District": political.admin2, "Location precision": political.geo_precision,
            "Date precision": political.time_precision}
names = list(cat_vars)
V = pd.DataFrame(np.nan, index=names, columns=names); sparse = V.copy()
for i, a in enumerate(names):
    for b in names[: i + 1]:
        if a == b:
            V.loc[a, b] = 1.0
        else:
            V.loc[a, b], sparse.loc[a, b] = cramers_v(cat_vars[a], cat_vars[b])

fig, axes = plt.subplots(1, 2, figsize=(16, 6.2), gridspec_kw={"width_ratios": [1, 1.05]})
sns.heatmap(rho, mask=np.triu(np.ones_like(rho, dtype=bool), k=1), annot=annot_r, fmt="", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Spearman rho"}, ax=axes[0])
for a_ in axes:
    a_.grid(False)
axes[0].set_xticklabels([wk_lab[c] for c in wk_cols], rotation=35, ha="right")
axes[0].set_yticklabels([wk_lab[c] for c in wk_cols], rotation=0)
axes[0].set_title(f"Which weekly Gaza measures move together?\n(Spearman, {len(wk)} complete weeks; † built-in overlap, ‡ serial dependence)")
sns.heatmap(V, mask=np.triu(np.ones_like(V, dtype=bool), k=1), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Cramér's V"}, ax=axes[1])
axes[1].set_xticklabels(names, rotation=35, ha="right"); axes[1].set_yticklabels(names, rotation=0)
axes[1].set_title(f"Which categorical fields are associated?\n(Cramér's V, {len(political):,} political-violence records)")
plt.tight_layout()
plt.show()

print("Share of contingency-table cells with expected count < 5 (sparse cells can inflate V)")
display((sparse * 100).round(0).astype(object).where(sparse.notna(), ""))
print("Weekly pairwise sample sizes")
display(n_pair)


**Interpretation.** Weekly focal counts have Spearman correlations of 0.70 with the previous week and 0.66 with the preceding four-week total, while focal status is associated with sub-event type (V=0.56), event type (0.32) and precision (0.28). These values motivate history and quality checks, but overlapping definitions, time dependence and sparse categories limit what they establish.


The comparisons identify differences worth examining more carefully. Next, the chart register provides navigation, and Task 4 tests the period and event-type questions while keeping effect size and assumptions visible.


# Task 3 - Main figure register

The main figures follow the sequence from data description to comparisons, tests, temporal patterns and spatial sensitivity. Supporting figures are linked where relevant and collected after the closing discussion, before References.


| Main figure | Content |
|---|---|
| M1 | [Fatality distributions](#1.2-Reported-fatalities---common-values-and-rare-extremes) |
| M2 | [Event/sub-event composition](#1.3-Event-and-sub-event-types) |
| M3 | [Recording precision](#1.5-Location-and-date-precision) |
| M4 | [Fatality occurrence and conditional totals](#2.3-How-often-is-a-fatality-recorded,-and-how-large-is-the-total?) |
| M5 | [Precision and focal prevalence](#2.4-Does-location-precision-change-the-focal-share?) |
| M6 | [Actor/sub-event combinations](#2.5-Which-actor-labels-and-sub-event-types-occur-together?) |
| M7 | [Weekly correlation and categorical association](#2.10-Which-measures-move-together,-and-which-categories-are-associated?) |
| M8 | [Period rates, shares and Hypothesis A](#Figure-M8---Period-rates,-focal-shares-and-within-type-evidence) |
| M9 | [Event types and Hypothesis B](#4.3-Hypothesis-B---Does-focal-share-differ-by-event-type?) |
| M10 | [Weekly counts, anomaly markers and focal share](#5.3-Which-Gaza-wide-weeks-stand-out?) |
| M11 | [Calendar patterns and persistence](#Figure-M11---Calendar-patterns-versus-persistence) |
| M12 | [District counts and variability](#Figure-M12---District-counts,-typical-volume-and-variability) |
| M13 | [Spatial maps and concentration](#Figure-M13---Spatial-concentration-and-precision-sensitivity) |
| M14 | [Spatial-setting sensitivity](#Figure-M14---How-sensitive-is-spatial-clustering-to-its-settings?) |

The main report has **14 figure groups**. Related existing plots share panels; **13 supporting figures** remain in the [appendix](#Appendix---Supporting-figures), including actor marginals, all-event totals, additional severity/share plots, minority counts, high-fatality review, monthly/weekday summaries, K-means and coordinate precision bars. Source tables remain beside their calculations; no new statistical analysis is introduced by the layout.


# Task 4 - Test the period and event-type differences

The earlier comparisons describe the observed records. Here we ask whether their category counts depart from what a simple equal-share model would predict, while recognizing that events are related across time and place.


## 4.1 Hypotheses and how to read the tests

**Source:** political-violence records from `acled_events_clean.csv`; one observation is one coded event.

| Item | Hypothesis A: period | Hypothesis B: event type |
|---|---|---|
| Null hypothesis (H0) | Focal share is the same before and from October 7 | Focal share is the same across the five types |
| Alternative (M8) | The two shares differ | At least one type's share differs |
| Primary method | Pearson chi-square on a 2 x 2 count table | Chi-square statistic with a Monte Carlo p-value on a 5 x 2 table |
| Sparse-count check | Smallest expected count 183.17 | Smallest expected count 3.30; four of ten cells below 5 |
| Sensitivity checks | Fisher exact, continuity correction and within-type comparisons | Pool the two rare types; compare the two focal-containing types |

An **expected count** is what the equal-share model predicts for a table cell. A **p-value** measures how unusual the observed statistic would be under that model and its assumptions; it is not the probability that H0 is true. Monte Carlo uses 99,999 simulated tables with the same margins and a fixed seed; its smallest reported estimate here is 0.00001.

We use **0.025** as the decision threshold for each of two primary tests (0.05 divided by two, the Bonferroni adjustment). Effect sizes show the magnitude: percentage-point difference, share ratio (called a risk ratio in the output), odds ratio and phi/V; 95% intervals show uncertainty under the model. These ratios concern recorded-event classification, not an individual's risk of harm.

Large expected counts support the usual chi-square approximation but do not establish independence. The tests and intervals treat records as independent even though events can cluster, so their inferential calibration is limited; we use them as exploratory evidence and do not claim causation. 'Pearson chi-square' is a categorical association test, not Pearson's numeric correlation coefficient.


The helpers below build the count tables, check expected counts, compute Wilson intervals for proportions and format p-values without claiming precision below the simulation limit.


In [ ]:
def focal_table(frame, by, order):
    """Focal / not-focal counts for each group in `order`. reindex keeps empty groups, and it reads
    [True, False] as column labels (a [True, False] list passed to .loc would be read as a boolean mask)."""
    tab = pd.crosstab(frame[by], frame.focal_event).reindex(index=order, columns=[True, False], fill_value=0)
    tab.columns = ["Focal", "Not focal"]
    return tab

def expected_counts(table):
    """Expected counts under independence: row total x column total / grand total."""
    t = np.asarray(table, dtype=float)
    return np.outer(t.sum(axis=1), t.sum(axis=0)) / t.sum()

def preflight(label, table):
    """Print the expected-count diagnostic; this does not check independence."""
    e = expected_counts(table)
    print(f"{label}: smallest expected count = {e.min():.2f}; cells below 5 = {(e < 5).sum()} of {e.size}")
    return e

def wilson_ci(k, n):
    """95% Wilson interval for a share; it stays inside 0 to 1 and works when k = 0 or k = n."""
    ci = binomtest(int(k), int(n)).proportion_ci(confidence_level=0.95, method="wilson")
    return ci.low, ci.high

def fmt_p(p, monte_carlo=False):
    """Readable p-value. Floating point underflows to 0.0 for very small chi-square p-values, and a
    Monte Carlo p-value cannot go below 1 / (N_RESAMPLES + 1)."""
    if monte_carlo:
        floor = 1 / (N_RESAMPLES + 1)
        return f"p = {floor:.5f} (simulation floor)" if np.isclose(p, floor) else f"p = {p:.5f}"
    return "p < 1e-300" if p == 0 else f"p = {p:.2e}"


## 4.2 Hypothesis A - Is focal share different between periods?

We test the two period shares with Pearson chi-square without continuity correction, since expected counts are large, and show Fisher exact and corrected results as sensitivity checks. Period length matters for the earlier weekly rates, but a proportion uses the records within its own period as denominator.


In [ ]:
period_table = focal_table(political, "period", [BEFORE, AFTER])
display(period_table.assign(Total=period_table.sum(axis=1),
                            Focal_share=(period_table.Focal / period_table.sum(axis=1)).round(3)))
preflight("A: period x focal", period_table)

tab_A = period_table.to_numpy()
chi2_A, p_A, dof_A, _ = chi2_contingency(tab_A, correction=False)   # Pearson chi-square, no continuity correction
p_A_yates = chi2_contingency(tab_A, correction=True)[1]             # sensitivity: with continuity correction
p_A_fisher = fisher_exact(tab_A)[1]                                 # sensitivity: exact test
n_A = tab_A.sum()
phi_A = np.sqrt(chi2_A / n_A)

# Effect sizes with 95% intervals (normal approximation; the ratios use the log scale)
(f_b, nf_b), (f_a, nf_a) = tab_A
n_b, n_a = f_b + nf_b, f_a + nf_a
share_b, share_a = f_b / n_b, f_a / n_a
diff = share_a - share_b
se_diff = np.sqrt(share_b * (1 - share_b) / n_b + share_a * (1 - share_a) / n_a)
rr = share_a / share_b
se_rr = np.sqrt((1 - share_a) / f_a + (1 - share_b) / f_b)
odds = (f_a * nf_b) / (f_b * nf_a)
se_or = np.sqrt(1 / f_a + 1 / nf_a + 1 / f_b + 1 / nf_b)

print(f"Pearson chi-square({dof_A}) = {chi2_A:,.2f}, {fmt_p(p_A)} (n = {n_A:,}, phi = {phi_A:.3f})")
print(f"Sensitivity: with continuity correction {fmt_p(p_A_yates)}; Fisher exact {fmt_p(p_A_fisher)}")
print(f"Focal share: {share_b:.1%} ({f_b:,} / {n_b:,}) before vs. {share_a:.1%} ({f_a:,} / {n_a:,}) from Oct 7 onward")
print(f"Difference = {diff * 100:+.1f} points (95% CI {(diff - Z * se_diff) * 100:+.1f} to {(diff + Z * se_diff) * 100:+.1f}); "
      f"risk ratio = {rr:.2f} ({rr * np.exp(-Z * se_rr):.2f} to {rr * np.exp(Z * se_rr):.2f}); "
      f"odds ratio = {odds:.2f} ({odds * np.exp(-Z * se_or):.2f} to {odds * np.exp(Z * se_or):.2f})")
decision_A = "Reject H0" if p_A < ALPHA / 2 else "Fail to reject H0"
print(f"Decision: {decision_A} (alpha = {ALPHA}, Bonferroni-adjusted to {ALPHA / 2} for the two primary tests)")


**Test result.** The test rejects equal shares (chi-square=151.60, p about 7.8e-35): the observed difference is 31.4 percentage points, with a model-based 95% interval of 27.7-35.1 points. The small phi value (0.082) coexists with a large share difference because the two period groups are highly unequal in size; neither statistic resolves changes in recording coverage.


### Figure M8 - Period rates, focal shares and within-type evidence

Top: the existing calendar-adjusted weekly rate plot (log axis). Bottom left: the overall focal shares with 95% Wilson intervals; bottom right: the same period comparison inside the two types containing focal records. All panels use the event master's political records; shares and rates have different denominators and units.


In [ ]:
# Within each event type that contains focal events: is the period difference still there?
within_tabs, within_rows = {}, []
for etype in EVENT_TYPES[:2]:
    t = focal_table(political.loc[political.event_type == etype], "period", [BEFORE, AFTER]).to_numpy()
    within_tabs[etype] = t
    e_min = expected_counts(t).min()
    if e_min >= 5:   # expected-count check for the large-sample approximation
        test_name, p_val = "Chi-square", chi2_contingency(t, correction=False)[1]
    else:
        test_name, p_val = "Fisher exact", fisher_exact(t)[1]
    within_rows.append({"Event type": etype,
                        "Before: focal / records": f"{t[0, 0]:,} / {t[0].sum():,}", "Before share": t[0, 0] / t[0].sum(),
                        "After: focal / records": f"{t[1, 0]:,} / {t[1].sum():,}", "After share": t[1, 0] / t[1].sum(),
                        "Smallest expected": e_min, "Test": test_name, "p-value": fmt_p(p_val)})
display(pd.DataFrame(within_rows).set_index("Event type").round({"Before share": 3, "After share": 3, "Smallest expected": 2}))


def draw_period_shares(fig, axes):
    # Left: the tested comparison, with 95% Wilson intervals
    ax = axes[0]
    shares = np.array([share_b, share_a])
    cis = np.array([wilson_ci(f_b, n_b), wilson_ci(f_a, n_a)])
    bars = ax.bar([BEFORE, AFTER], shares, color=[C_GREY, C_FOCAL], width=0.55,
                  yerr=[shares - cis[:, 0], cis[:, 1] - shares], capsize=5, error_kw={"lw": 1.2})
    for b, k, n, s, hi in zip(bars, (f_b, f_a), (n_b, n_a), shares, cis[:, 1]):
        ax.text(b.get_x() + b.get_width() / 2, hi + 0.015, f"{k:,} / {n:,}\n{s:.1%}", ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, 0.7)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Period")
    ax.set_ylabel("Focal events / political-violence records")
    ax.set_title(f"Hypothesis A: did the focal share change?\n(chi-square {fmt_p(p_A)}; bars show 95% intervals)")
    
    # Right: the same comparison inside the two event types that contain focal events
    ax = axes[1]
    xs, w = np.arange(2), 0.36
    for j, (period, col) in enumerate([(BEFORE, C_GREY), (AFTER, C_FOCAL)]):
        vals, lows, highs, notes = [], [], [], []
        for etype in EVENT_TYPES[:2]:
            k, n = within_tabs[etype][j, 0], within_tabs[etype][j].sum()
            lo, hi = wilson_ci(k, n)
            vals.append(k / n); lows.append(k / n - lo); highs.append(hi - k / n)
            notes.append(f"{k:,}/{n:,}\n{k / n:.0%}" + ("\n(n < 30)" if n < 30 else ""))
        bars = ax.bar(xs + (j - 0.5) * w, vals, w, color=col, label=period, yerr=[lows, highs], capsize=4, error_kw={"lw": 1.1})
        for b, v, hi, note in zip(bars, vals, highs, notes):
            ax.text(b.get_x() + b.get_width() / 2, v + hi + 0.015, note, ha="center", va="bottom", fontsize=8.5)
    ax.set_xticks(xs)
    ax.set_xticklabels(["Explosions/\nRemote violence", "Violence against\ncivilians"])
    ax.set_ylim(0, 1.3); ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Event type")
    ax.set_ylabel("Focal events / political-violence records of that type")
    ax.set_title("Within these two event types, did the focal share change?")
    ax.legend(loc="upper left")


In [ ]:
fig = plt.figure(figsize=(15, 10), layout="constrained")
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.2])
draw_period_rates(fig, fig.add_subplot(grid[0, :]))
draw_period_shares(fig, [fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])])
fig.suptitle("Figure M8. Period comparison: rates, shares and within-type evidence", fontweight="bold")
plt.show()


**Interpretation.** Both the focal rate and share rise between periods, and the share increase also occurs within Explosions (13.1% to 49.4%), so the overall contrast is not solely a change in event-type mix. The earlier cohort is sparse and the intervals/tests assume independent records; reporting changes remain an unresolved explanation.


## 4.3 Hypothesis B - Does focal share differ by event type?

We retain all five political-violence event types and use Monte Carlo simulation because Protests and Riots have sparse expected counts. **Figure M9** shows each type's focal share with Wilson intervals and a residual heatmap: a positive residual means more records than the equal-share model expects, and a negative one means fewer. Residuals are descriptive contributions to the statistic, not separately corrected hypothesis tests.


In [ ]:
type_table = focal_table(political, "event_type", EVENT_TYPES)
display(type_table.assign(Total=type_table.sum(axis=1),
                          Focal_share=(type_table.Focal / type_table.sum(axis=1)).round(3)))
exp_B = preflight("B: event type x focal", type_table)

# The preflight fails (see above), so the p-value comes from Monte Carlo resampling, as in Phase 2.
# All five event types stay in the table.
tab_B = type_table.to_numpy()
resampling = MonteCarloMethod(n_resamples=N_RESAMPLES, rng=np.random.default_rng(SEED))
res_B = chi2_contingency(tab_B, correction=False, method=resampling)
chi2_B, p_B = res_B.statistic, res_B.pvalue
dof_B = (tab_B.shape[0] - 1) * (tab_B.shape[1] - 1)   # the Monte Carlo result does not report degrees of freedom
n_B = tab_B.sum()
V_B = np.sqrt(chi2_B / (n_B * (min(tab_B.shape) - 1)))

print(f"Pearson chi-square({dof_B}) = {chi2_B:,.2f}, Monte Carlo {fmt_p(p_B, monte_carlo=True)} "
      f"({N_RESAMPLES:,} simulated tables with the same row and column totals)")
print(f"Cramér's V = {V_B:.3f} (n = {n_B:,})")
decision_B = "Reject H0" if p_B < ALPHA / 2 else "Fail to reject H0"
print(f"Decision: {decision_B} (alpha = {ALPHA}, Bonferroni-adjusted to {ALPHA / 2} for the two primary tests)")

# Which cells drive the statistic? Pearson residuals: (observed - expected) / sqrt(expected)
resid_B = pd.DataFrame((tab_B - exp_B) / np.sqrt(exp_B), index=EVENT_TYPES, columns=["Focal", "Not focal"])
print("\nPearson residuals (values beyond about +/-2 stand out; very large values mean far from independence)")
display(resid_B.round(1))

# Sensitivity 1: pool the two rare types so every expected count is at least 5, then use the ordinary test
pooled = type_table.copy()
pooled.loc["Protests and Riots (pooled)"] = pooled.loc[["Protests", "Riots"]].sum()
pooled = pooled.drop(["Protests", "Riots"])
preflight("B with Protests and Riots pooled (15 records)", pooled)
chi2_pool, p_pool, dof_pool, _ = chi2_contingency(pooled.to_numpy(), correction=False)
print(f"Pooled table: chi-square({dof_pool}) = {chi2_pool:,.2f}, {fmt_p(p_pool)}")

# Sensitivity 2: the only two event types that contain focal events, compared directly
pair = type_table.loc[EVENT_TYPES[:2]].to_numpy()
chi2_pair, p_pair, _, e_pair = chi2_contingency(pair, correction=False)
print(f"Violence against civilians vs. Explosions/Remote violence: chi-square(1) = {chi2_pair:,.2f}, {fmt_p(p_pair)} "
      f"(smallest expected count = {e_pair.min():.2f})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.5, 1]})

# Left: focal share within each event type, with 95% Wilson intervals; groups with fewer than 30 records are hatched
ax = axes[0]
tb = type_table.assign(n=type_table.sum(axis=1))
tb["share"] = tb.Focal / tb.n
tb = tb.sort_values("share")
small = (tb.n < 30).to_numpy()
cis_B = np.array([wilson_ci(k, n) for k, n in zip(tb.Focal, tb.n)])
bars = ax.barh(tb.index, tb.share, color=[C_GREY if s else C_FOCAL for s in small],
               xerr=[tb.share - cis_B[:, 0], cis_B[:, 1] - tb.share], capsize=4, error_kw={"lw": 1.1})
for b, s in zip(bars, small):
    if s:
        b.set_hatch("//")
for y, (_, r) in enumerate(tb.iterrows()):
    note = "  (n < 30)" if r.n < 30 else ""
    ax.text(cis_B[y, 1] + 0.012, y, f"{r.Focal:,.0f} / {r.n:,.0f} = {r.share:.1%}{note}", va="center", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
overall = tab_B[:, 0].sum() / n_B
ax.axvline(overall, color="black", ls="--", lw=1)
ax.text(overall + 0.008, 0.5, f"all political-violence\nrecords: {overall:.1%}", fontsize=8.5, va="center",
        bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
ax.grid(axis="x")
ax.set_xlim(0, 1.5); ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Focal events / political-violence records of that event type")
ax.set_ylabel("Event type")
ax.set_title(f"Hypothesis B: does the focal share differ by event type?\n(Monte Carlo {fmt_p(p_B, monte_carlo=True)}; Cramér's V = {V_B:.2f})")

# Right: which cells push the table away from independence
ax = axes[1]
lim = float(np.abs(resid_B.to_numpy()).max())
sns.heatmap(resid_B, annot=True, fmt=".1f", cmap="RdBu_r", vmin=-lim, vmax=lim, center=0,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Pearson residual"}, ax=ax)
ax.grid(False)
ax.set_xlabel("Record status")
ax.set_ylabel("")
ax.set_title("Which cells differ most from what independence predicts?")
plt.tight_layout()
plt.show()


**Interpretation.** Equal focal shares are rejected (chi-square=2,394.76, Monte Carlo estimate **p=0.00001**, the simulation floor, V=0.32): Violence against civilians is 99.3% focal, Explosions 48.9%, and Battles 0%, with the largest residual departures in the first and last of those groups. Pooling rare types and comparing the two focal-containing types retain the conclusion, but the civilian-actor requirement makes part of this association structural and the test does not establish causation.


## 4.4 Test summary

This compact table collects the computed statistics rather than introducing another comparison. Both primary decisions use the adjusted 0.025 threshold.


In [ ]:
summary = pd.DataFrame([
    {"Hypothesis": "A: focal share, before vs. after Oct 7, 2023", "BQ": "BQ1",
     "Test": "Pearson chi-square, 2 × 2", "Statistic": f"chi-square({dof_A}) = {chi2_A:,.2f}", "p-value": fmt_p(p_A),
     "Effect size": f"{diff * 100:+.1f} points; odds ratio {odds:.2f}; phi = {phi_A:.2f}", "Decision": decision_A},
    {"Hypothesis": "B: focal share across event types", "BQ": "BQ2",
     "Test": f"Chi-square, Monte Carlo p ({N_RESAMPLES:,} tables)", "Statistic": f"chi-square({dof_B}) = {chi2_B:,.2f}",
     "p-value": fmt_p(p_B, monte_carlo=True), "Effect size": f"Cramér's V = {V_B:.2f}", "Decision": decision_B},
]).set_index("Hypothesis")
print(f"Decision rule: reject H0 when p < {ALPHA / 2} (alpha = {ALPHA}, Bonferroni-adjusted for two primary tests)")
display(summary)


**What the tests establish.** Period and event type are associated with focal classification in this snapshot, and the period difference also occurs within Explosions. They do not distinguish activity from reporting changes or verify actor roles; next we examine how concentrated, persistent and unusual the recorded patterns are.


# Task 5 - Which patterns deserve closer review?

This task moves from group differences to practical review questions: where comparisons have few examples, which records or weeks stand out, whether patterns repeat, and how events concentrate in time and space. Each method is a descriptive screening tool, not a verified label of an important or harmful event.


### How unusual weeks are screened

A **median** describes the middle week; **MAD**, the median absolute deviation from that median, describes typical variation. A modified z-score compares a week with that baseline: `0.6745 x (count - median) / MAD`; values beyond +/-3.5 are flagged for review, not declared errors ([method reference](#Outlier-screening-method)).

We compare the whole complete period with a rolling baseline using only the **previous eight calendar weeks**. Counts use a minimum MAD of one event to avoid division by zero; shares use their own MAD with no one-event floor, and a flat share history is left unscored. Missing/ineligible history is never bridged by skipping weeks.


In [ ]:
Z_CUT = 3.5

def robust_z(s):
    """Descriptive whole-series modified z-score; flat series are unscored."""
    med = s.median()
    mad = (s - med).abs().median()
    return 0.6745 * (s - med) / mad if mad > 0 else pd.Series(np.nan, index=s.index)

def trailing_robust_z(s, window=8, mad_floor=1.0):
    """Previous calendar weeks only; any missing current/history value leaves no score."""
    vals = s.to_numpy(float)
    out = np.full(len(vals), np.nan)
    for i in range(window, len(vals)):
        hist = vals[i-window:i]
        if not np.isfinite(vals[i]) or not np.isfinite(hist).all():
            continue
        med = np.median(hist)
        mad = max(float(np.median(np.abs(hist-med))), mad_floor)
        if mad > 0:
            out[i] = 0.6745 * (vals[i]-med) / mad
    return pd.Series(out, index=s.index)


## 5.1 Are some groups too small for reliable comparison?

**Sources:** political records from the master, plus zero-count summaries from both weekly files. The table contrasts focal and non-focal examples in each slice; **Figure A6** focuses on the smaller class, adding information beyond the earlier share plots. A minority count of zero means that class is absent, not merely uncommon.


In [ ]:
def imbalance_row(label, df):
    n = len(df); k = int(df.focal_event.sum()); minority = min(k, n - k)
    return {"Slice": label, "Records": n, "Focal": k, "Not focal": n - k,
            "Focal share": k / n, "Minority class": "focal" if k < n - k else "not focal",
            "Minority count": minority, "Majority : minority": (max(k, n - k) / minority) if minority else np.nan}

post = political[political.period == AFTER]
slices = [
    ("All political-violence records", political),
    ("Before Oct 7, 2023", political[political.period == BEFORE]),
    ("Oct 7, 2023 onward", post),
    ("Oct 7+ · precision 1", post[post.geo_precision == 1]),
    ("Oct 7+ · precision 2", post[post.geo_precision == 2]),
    ("Oct 7+ · Explosions/Remote violence", post[post.event_type == "Explosions/Remote violence"]),
    ("Oct 7+ · Violence against civilians", post[post.event_type == "Violence against civilians"]),
    ("Oct 7+ · Battles", post[post.event_type == "Battles"]),
]
imb = pd.DataFrame([imbalance_row(l, d) for l, d in slices]).set_index("Slice")
display(imb.round({"Focal share": 3, "Majority : minority": 2}))

# Weekly view of the same question: how often is a week "quiet" for focal events?
print("Weeks in the Gaza-week table:", len(gaza_week),
      "| weeks with zero political-violence records:", int((gaza_week.political_events == 0).sum()),
      "| weeks with zero focal events:", int((gaza_week.focal_events == 0).sum()))
dw_zero = district_week.assign(zero_focal=district_week.focal_events == 0,
                               zero_political=district_week.political_events == 0)
display(dw_zero.groupby("admin2")[["zero_focal", "zero_political"]].sum().rename(
    columns={"zero_focal": "district-weeks with 0 focal events", "zero_political": "district-weeks with 0 political-violence records"}))


### Figure A6 - Size of the smaller class

The horizontal axis uses a symmetric-log scale with a linear region near zero, so zero remains zero and very different sample sizes fit on one display. Shares and majority/minority ratios remain in the preceding table.


In [ ]:
plot_df = imb.iloc[::-1]
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(plot_df.index, plot_df["Minority count"], color=C_OTHER)
for y, (_, r) in enumerate(plot_df.iterrows()):
    ax.annotate(f"{r['Minority count']:,.0f} ({r['Minority class']})",
                (r["Minority count"], y), xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_xscale("symlog", linthresh=1)
ax.set_xlim(0, 100000)
ax.set_xlabel("Records in the smaller class (symlog scale; zero shown at zero)")
ax.set_ylabel("Political-violence record subset")
ax.set_title("How many examples does the smaller class have?")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); save_supporting_figure("A6", fig)


The supporting figure and its interpretation are in [Appendix A6](#A6---Smaller-class-sample-sizes); the calculations and tables above remain available here.


## 5.2 Which high-fatality records need review?

**Source:** `acled_events_clean.csv`, all periods; `high_fatalities_review` selects reported totals of 24 or more. [Figure M1](#1.2-Reported-fatalities---common-values-and-rare-extremes) already establishes the overall distribution, so this section adds record identity, flagged-group composition and timing rather than repeating that distribution.


In [ ]:
f = master.fatalities
thr = int(master.loc[master.high_fatalities_review, "fatalities"].min())
thr_below = int(master.loc[~master.high_fatalities_review, "fatalities"].max())
flag = master[master.high_fatalities_review]
top1 = master.nlargest(int(len(master) * 0.01), "fatalities")

review_detail = pd.Series({
    "Gaza IQR upper fence (Q3 + 1.5 x IQR)": f.quantile(.75) + 1.5 * (f.quantile(.75)-f.quantile(.25)),
    "Records selected in top 1% (floor of 1% x n)": len(top1),
    "Share of reported fatalities in that top 1%": top1.fatalities.sum() / f.sum(),
    "Flagged records that are focal": int(flag.focal_event.sum()),
}).to_frame("value")
display(review_detail)
print("Flagged records by event type and focal status")
display(flag.groupby(["event_type", "focal_event"]).size().rename("flagged records").to_frame())
print("Flagged records by location precision:", flag.geo_precision.value_counts().sort_index().to_dict())


### Figure A7 - The largest recorded fatality totals

These are individual event records, identified by date and location. The table retains IDs and sources so the numbers can be checked against the underlying reports.


In [ ]:
top = master.nlargest(12, "fatalities").iloc[::-1]
labels = [f"{d:%b %d, %Y} · {loc}" for d, loc in zip(top.event_date, top.location)]

display(top.iloc[::-1][["event_id_cnty", "event_date", "location", "fatalities", "source"]])
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(top)), top.fatalities, color=[C_FOCAL if v else C_GREY for v in top.focal_event])
ax.set_yticks(range(len(top))); ax.set_yticklabels(labels, fontsize=8.5)
for y, v in enumerate(top.fatalities):
    ax.text(v + 4, y, f"{v}", va="center", fontsize=8.5)
ax.set_xlabel("Reported fatalities in one record")
ax.set_title("Which 12 records report the most fatalities? (red = focal, grey = not focal)")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); save_supporting_figure("A7", fig)


The supporting figure and its interpretation are in [Appendix A7](#A7---Highest-reported-fatality-records); the calculations and tables above remain available here.


### Figure A8 - When are flagged records concentrated?

**Source:** the master's high-fatality subset, grouped by event-calendar quarter and focal status. The table also divides flagged records by all event records in that quarter; the chart shows counts from Q4 2023 onward, when flagged records occur.


In [ ]:
q = flag.assign(quarter=flag.event_date.dt.to_period("Q").astype(str))
qt = pd.crosstab(q.quarter, q.focal_event).reindex(columns=[True, False], fill_value=0)
qt.columns = ["Focal", "Not focal"]
all_q = master.assign(quarter=master.event_date.dt.to_period("Q").astype(str))
qt = qt.reindex(sorted(all_q.quarter.unique()), fill_value=0)
qt = qt.loc[qt.index >= "2023Q4"]
qt["Share of that quarter's records flagged"] = qt.sum(axis=1) / all_q.groupby("quarter").size().reindex(qt.index)
display(qt.round({"Share of that quarter's records flagged": 3}))

fig, ax = plt.subplots(figsize=(10.5, 4.6))
ax.bar(qt.index, qt["Focal"], color=C_FOCAL, label="Focal")
ax.bar(qt.index, qt["Not focal"], bottom=qt["Focal"], color=C_GREY, label="Not focal")
for x, (_, r) in enumerate(qt.iterrows()):
    ax.text(x, r["Focal"] + r["Not focal"] + 4, f"{int(r['Focal'] + r['Not focal'])}", ha="center", fontsize=9)
ax.set_xlabel("Quarter of event date"); ax.set_ylabel("Records flagged for fatality review")
ax.set_title("When are the high-fatality records concentrated?")
ax.legend(frameon=False); plt.tight_layout(); save_supporting_figure("A8", fig)


The supporting figure and its interpretation are in [Appendix A8](#A8---Timing-of-high-fatality-flags); the calculations and tables above remain available here.


## 5.3 Which Gaza-wide weeks stand out?

**Source:** `acled_gaza_week.csv`, 103 complete weeks; all of these meet Phase 2's political-activity history rule. We compare whole-period scores with the preceding-eight-week rule; the first eight weeks lack a trailing score, and the final partial week is shown but not scored. Both counts and shares are screened, with the scale-specific MAD handling explained above.


In [ ]:
gwc = gaza_week[gaza_week.complete_week].sort_values("week_start").set_index("week_start").copy()
partial = gaza_week[~gaza_week.complete_week]
for c in ["focal_events", "political_events", "focal_share"]:
    gwc[f"rz_{c}"] = robust_z(gwc[c])                    # against the whole period
    gwc[f"tz_{c}"] = trailing_robust_z(gwc[c], window=8, mad_floor=0.0 if c == "focal_share" else 1.0)  # against the previous 8 weeks only

med = {c: gwc[c].median() for c in ["focal_events", "political_events", "focal_share"]}
mad = {c: (gwc[c] - med[c]).abs().median() for c in med}
print("Median / MAD of weekly values (complete weeks):",
      {c: (round(med[c], 3), round(mad[c], 3)) for c in med})
print(f"|modified z| > {Z_CUT}:  focal_events {int((gwc.rz_focal_events.abs() > Z_CUT).sum())} weeks | "
      f"political_events {int((gwc.rz_political_events.abs() > Z_CUT).sum())} | focal_share {int((gwc.rz_focal_share.abs() > Z_CUT).sum())}")
print(f"Trailing 8-week rule: focal_events {int((gwc.tz_focal_events.abs() > Z_CUT).sum())} weeks | "
      f"political_events {int((gwc.tz_political_events.abs() > Z_CUT).sum())} | focal_share {int((gwc.tz_focal_share.abs() > Z_CUT).sum())}")

flagged = gwc[(gwc[["rz_focal_events", "rz_political_events", "tz_focal_events", "tz_political_events"]].abs() > Z_CUT).any(axis=1)]
display(flagged[["focal_events", "political_events", "focal_share", "rz_focal_events", "rz_political_events",
                 "tz_focal_events", "tz_political_events"]].round(2))


### Figure M10 - Unusual weekly counts and their focal share

**Source:** Gaza-week CSV; top two panels reuse the focal/political count plots with whole-period and prior-eight-week anomaly markers, while the bottom reuses the weekly focal-share panel. The bottom dashed line is the master-derived pooled post-cutoff share, the shaded interval marks the five lowest focal-count weeks, and hollow markers identify the partial week. Count flags and focal shares answer different questions even though they share a timeline.


In [ ]:
def draw_weekly_anomalies(fig, axes):
    for ax, col, title, ylab in [
        (axes[0], "focal_events", "Which weeks have unusual numbers of focal events?", "Focal events per week"),
        (axes[1], "political_events", "Which weeks have unusual numbers of political-violence records?", "Political-violence records per week")]:
        s = gwc[col]
        ax.plot(s.index, s, color=C_OTHER, lw=1.3, label="Weekly count (complete weeks)")
        band = 1.4826 * mad[col] * Z_CUT
        ax.axhspan(med[col] - band, med[col] + band, color=C_GREY, alpha=0.18, label=f"Median ± {Z_CUT} robust SD (whole period)")
        ax.axhline(med[col], color=C_GREY, lw=1)
        g_flag = gwc[gwc[f"rz_{col}"].abs() > Z_CUT]
        t_flag = gwc[(gwc[f"tz_{col}"].abs() > Z_CUT) & ~(gwc[f"rz_{col}"].abs() > Z_CUT)]
        ax.scatter(g_flag.index, g_flag[col], color=C_FOCAL, s=45, zorder=3, label="Flagged against the whole period")
        ax.scatter(t_flag.index, t_flag[col], color=C_WARN, marker="D", s=40, zorder=3, label="Flagged only against previous 8 weeks")
        ax.scatter(partial.week_start, partial[col], facecolors="none", edgecolors="black", s=55, zorder=4, label="Incomplete final week (not scored)")
        ax.set_title(title); ax.set_ylabel(ylab)
        ax.legend(frameon=False, ncol=2, fontsize=8, loc="upper right")
    axes[1].set_xlabel("Week start")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True, layout="constrained")
draw_weekly_anomalies(fig, axes[:2])
draw_weekly_share(fig, axes[2])
fig.suptitle("Figure M10. Weekly counts, anomaly screens and focal share", fontweight="bold")
plt.show()


**Interpretation.** The trailing count rule flags several rises following the early-2025 trough, while the focal share can remain high during low-count weeks because its political denominator also falls. These are screening leads sensitive to recent history, not explanations of the dip or verified anomalies.

**Share-screening check.** The whole-period share rule flags no weeks and the trailing share rule flags 11; these are separate from the count flags shown above. Flat share-history windows remain unscored.


### District-week screening and baseline sensitivity

**Source:** `acled_district_week.csv`, complete weeks. Primary scores use the saved `focal_count_for_history`, requiring an eligible current week and eight eligible preceding calendar weeks; North Gaza's two complete zero-political weeks and affected windows therefore remain unscored. The comparison table also retains the original all-complete-count policy as a sensitivity check, making its effect visible.


In [ ]:
# Same trailing rule by district; primary scores require eligible current and prior weeks.
dwc = district_week[district_week.complete_week].sort_values(["admin2", "week_start"]).copy()
dwc["tz_focal"] = dwc.groupby("admin2").focal_count_for_history.transform(lambda s: trailing_robust_z(s, window=8))
dwc["tz_raw_complete"] = dwc.groupby("admin2").focal_events.transform(lambda s: trailing_robust_z(s, window=8))
baseline_sensitivity = pd.DataFrame([
    {"Baseline policy": label, "Scored district-weeks": int(dwc[col].notna().sum()),
     "Flagged district-weeks": int(dwc[col].abs().gt(Z_CUT).sum())}
    for label, col in [("Phase 2 eligible history (primary)", "tz_focal"),
                       ("All complete-week counts (sensitivity)", "tz_raw_complete")]
])
display(baseline_sensitivity)
elig = dwc.tz_focal.notna()
dist_flags = (dwc[elig].assign(high=lambda d: d.tz_focal > Z_CUT, low=lambda d: d.tz_focal < -Z_CUT)
              .groupby("admin2").agg(scored_weeks=("tz_focal", "size"), high_weeks=("high", "sum"), low_weeks=("low", "sum")))
dist_flags["flagged share"] = (dist_flags.high_weeks + dist_flags.low_weeks) / dist_flags.scored_weeks
display(dist_flags.round({"flagged share": 3}))
print("All districts together:", int((dist_flags.high_weeks + dist_flags.low_weeks).sum()), "flagged of",
      int(dist_flags.scored_weeks.sum()), "scored district-weeks =",
      f"{(dist_flags.high_weeks + dist_flags.low_weeks).sum() / dist_flags.scored_weeks.sum():.1%}")
print("Highest-scoring district-weeks (upward):")
display(dwc.nlargest(6, "tz_focal")[["admin2", "week_start", "focal_events", "political_events", "tz_focal"]].round({"tz_focal": 2}))


**Interpretation.** With Phase 2's eligibility rule, 48 of 465 scored district-weeks are flagged (10.3%); retaining zero-political weeks in the baseline instead gives 50 of 475 (10.5%). The difference is confined to North Gaza's missing eligible history, and several of the highest upward scores elsewhere still follow the early-2025 trough, so both baseline policy and recovery effects matter.


### Data-quality checks behind the patterns

**Sources:** the event master, Gaza-week file and spatial incident file. These checks look for missing/duplicate identifiers and broad coordinate or coverage problems; they screen the files rather than verifying every real-world event.


In [ ]:
lat_ok = master.latitude.between(31.2, 31.6) & master.longitude.between(34.2, 34.6)
dq = pd.DataFrame([
    ("Duplicate event IDs in the master", int(master.duplicated("event_id_cnty").sum()), "No repeated IDs; this alone cannot exclude real-world event overlap."),
    ("Missing coordinates in the master", int(master[["latitude", "longitude"]].isna().any(axis=1).sum()), "None missing."),
    ("Coordinates outside a rough Gaza box (lat 31.2-31.6, lon 34.2-34.6)", int((~lat_ok).sum()), "All inside the rough box; this does not verify each location."),
    ("Events with no district (kept in Gaza-wide totals)", int(master.admin2.isna().sum()), "Left out of district tables only."),
    ("Coast-coded events (kept in Gaza-wide totals)", int(master.coastal_location.sum()), "Left out of the land spatial table only."),
    ("Zero-event weeks in the Gaza-week table", int((gaza_week.recorded_events == 0).sum()), "None: weekly activity never stops in the record."),
    ("Incomplete final week", f"{partial.week_start.dt.date.iloc[0]} to {partial.week_end.dt.date.iloc[0]}", "Marked and left out of scoring; the export ends Sep 30, 2025."),
    ("Records with 0 reported fatalities", f"{(master.fatalities == 0).mean():.1%}", "A zero may mean no figure was recorded, not that nobody was hurt."),
    ("Records with time precision other than 1", f"{(master.time_precision != 1).mean():.1%}", "Lower date precision can move records between time bins; sensitivity is not tested here."),
    ("Land focal incidents", len(incidents), ""),
    ("Distinct coordinate pairs among them", len(incidents[["latitude", "longitude"]].drop_duplicates()), "Many incidents share one coordinate: they stand for named places, not exact points."),
], columns=["Check", "Result", "Reading"])
display(dq)


**Reading the quality checks.** IDs and coordinates are complete and the coordinates pass the rough geographic screen, but these checks do not establish that every real-world event is distinct or exactly located. Most records have date precision 1, yet date sensitivity remains untested, and repeated spatial coordinates describe named-place concentrations rather than exact incident points.


## 5.4 Is there a longer-term direction?

**Source:** `acled_gaza_week.csv`, complete weeks. Here a month receives all counts from weeks **starting in that month**, unlike the event-calendar-month totals in Task 1; first/last months and four/five-week months therefore have different exposure. The table reports monthly sums, number of weeks, quarterly weekly means and exploratory rank trends over time.


In [ ]:
wk = gwc.reset_index()
wk["month"] = wk.week_start.dt.to_period("M")
mo = wk.groupby("month").agg(focal=("focal_events", "sum"), political=("political_events", "sum"), weeks=("focal_events", "size"))
mo["share"] = mo.focal / mo.political
mo.index = mo.index.astype(str)
display(mo.round(3))

wk["quarter"] = wk.week_start.dt.to_period("Q").astype(str)
qtr = wk.groupby("quarter").agg(weeks=("focal_events", "size"), mean_focal_per_week=("focal_events", "mean"),
                                mean_political_per_week=("political_events", "mean"), pooled_share=("focal_events", "sum"))
qtr["pooled_share"] = qtr.pooled_share / wk.groupby("quarter").political_events.sum()
display(qtr.round({"mean_focal_per_week": 1, "mean_political_per_week": 1, "pooled_share": 3}))

# Rank trend against time (exploratory: weeks are serially dependent, so the p-values are only a rough guide)
t = np.arange(len(gwc))
trend = []
for c in ["focal_events", "political_events", "focal_share"]:
    rho, p = spearmanr(t, gwc[c])
    half = len(gwc) // 2
    trend.append({"Weekly measure": c, "Spearman rho vs week number": rho, "p (rough)": f"{p:.2e}",
                  "Median, first half": gwc[c].iloc[:half].median(), "Median, second half": gwc[c].iloc[half:].median()})
display(pd.DataFrame(trend).set_index("Weekly measure").round(3))


### Figure A9 - Monthly grouped counts and pooled share

Bars use the left count axis; the line uses the right share axis. The pooled share divides the monthly grouped focal sum by the matching political sum, rather than averaging weekly percentages; the two axes should not be compared by their vertical heights.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(mo))
ax.bar(x, mo.focal, color=C_FOCAL, alpha=0.85, label="Focal events (sum of complete weeks)")
ax.set_xticks(x); ax.set_xticklabels(mo.index, rotation=60, ha="right", fontsize=8.5)
ax.set_ylabel("Focal events per month"); ax.set_xlabel("Month of week start")
ax.set_title("Is there a lasting trend in monthly focal events and in their share?")
ax2 = ax.twinx()
ax2.plot(x, mo["share"], color=C_OTHER, marker="o", lw=1.8, label="Focal share of political-violence records")
ax2.set_ylabel("Focal share"); ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax2.set_ylim(0, 0.8)
ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, frameon=False, loc="upper right")
plt.tight_layout(); save_supporting_figure("A9", fig)


The supporting figure and its interpretation are in [Appendix A9](#A9---Monthly-grouped-count/share-summary); the calculations and tables above remain available here.


## 5.5 Calendar patterns and persistence

A recurring calendar pattern is different from persistence: the former repeats at a calendar interval, while the latter means nearby weeks resemble one another. We inspect both without assuming that either proves future predictability.


### Calendar comparison inputs

**Sources:** Gaza complete-week means by start-month/year and post-cutoff master focal records by weekday. The monthly comparison appears with autocorrelation in [Figure M11](#Figure-M11---Calendar-patterns-versus-persistence); the existing weekday plot is in [Appendix A11](#A11---Focal-records-by-weekday). The weekday counts include the final partial-week dates and are not exposure-adjusted rates.


In [ ]:
wk["year"], wk["mon"] = wk.week_start.dt.year, wk.week_start.dt.month
moy = wk.groupby(["year", "mon"]).focal_events.mean().unstack(0)
display(moy.round(1))
both = moy[[2024, 2025]].dropna()
rho_yy, _ = spearmanr(both[2024], both[2025])
print(f"Months present in both 2024 and 2025: {len(both)} (Jan-Sep). Spearman rho between the two years' monthly means = {rho_yy:.2f}")

post_focal = master[(master.focal_event) & (master.period == AFTER)]
dow = post_focal.event_date.dt.day_of_week.value_counts().sort_index()
dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print("Focal events by weekday:", dow.to_dict(), f"| highest / lowest = {dow.max() / dow.min():.2f}")


def draw_calendar_months(fig, ax):
    for yr, col in zip([2023, 2024, 2025], [C_GREY, C_OTHER, C_FOCAL]):
        s = moy[yr].dropna()
        ax.plot(s.index, s.values, marker="o", color=col, lw=1.8, label=str(yr) + (" (Oct-Dec only)" if yr == 2023 else " (Jan-Sep only)" if yr == 2025 else ""))
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
    ax.set_xlabel("Calendar month"); ax.set_ylabel("Mean focal events per complete week")
    ax.set_title("Do the same calendar months repeat across years?")
    ax.legend(frameon=False)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(dow.index, dow.values, color=C_OTHER)
ax.axhline(dow.mean(), color="black", ls="--", lw=1)
for i, v in enumerate(dow.values):
    ax.text(i, v + 15, f"{v:,}", ha="center", fontsize=8.5)
ax.set_ylim(0, dow.max() * 1.12)
ax.set_xlabel("Day of week (event date)"); ax.set_ylabel("Focal events, Oct 7, 2023 onward")
ax.set_title("Is any weekday busier for recorded focal events?")
plt.tight_layout(); save_supporting_figure("A11", fig)


The nine shared calendar months of 2024 and 2025 have rank correlation -0.30; that comparison is displayed with persistence in [Figure M11](#Figure-M11---Calendar-patterns-versus-persistence). Weekday counts vary by about 1.11 from highest to lowest and are shown separately in [Appendix A11](#A11---Focal-records-by-weekday).


### Figure M11 - Calendar patterns versus persistence

Top: the existing cross-year monthly comparison from complete Gaza weeks. Bottom: autocorrelation of focal counts and their week-to-week changes, using the same complete-week series; autocorrelation compares a sequence with itself at earlier lags. The shaded bands are approximate no-autocorrelation references, not tests of seasonal absence.


In [ ]:
def acf(x, nlags):
    x = np.asarray(x, float); x = x - x.mean()
    return np.array([(x[:-k] * x[k:]).sum() / (x * x).sum() for k in range(1, nlags + 1)])

series = gwc.focal_events.to_numpy(float)
lags = np.arange(1, 13)
acf_level, acf_diff = acf(series, 12), acf(np.diff(series), 12)
band = 1.96 / np.sqrt(len(series))
print("ACF of weekly focal events, lags 1-12:", np.round(acf_level, 2))
print("ACF of week-to-week changes, lags 1-12:", np.round(acf_diff, 2))
print(f"Approximate 95% band: ±{band:.2f}  (n = {len(series)} weeks).")
print("Annual repetition is not reliably established here: the table spans only", len(gaza_week), "weeks (about two years).")


def draw_persistence(fig, axes):
    for ax, vals, title in [(axes[0], acf_level, "Does a week resemble the weeks before it? (weekly focal events)"),
                            (axes[1], acf_diff, "After removing the level: do week-to-week changes repeat?")]:
        ax.bar(lags, vals, color=C_OTHER, width=0.6)
        ax.axhline(0, color="black", lw=0.8)
        ax.axhspan(-band, band, color=C_GREY, alpha=0.25, label="≈95% band if no autocorrelation")
        ax.set_xlabel("Lag (weeks)"); ax.set_title(title, fontsize=10)
        ax.set_xticks(lags)
    axes[0].set_ylabel("Autocorrelation"); axes[0].legend(frameon=False)


In [ ]:
fig = plt.figure(figsize=(15, 9), layout="constrained")
grid = fig.add_gridspec(2, 2)
draw_calendar_months(fig, fig.add_subplot(grid[0, :]))
acf_left = fig.add_subplot(grid[1, 0])
acf_right = fig.add_subplot(grid[1, 1], sharey=acf_left)
draw_persistence(fig, [acf_left, acf_right])
fig.suptitle("Figure M11. Calendar patterns and week-to-week persistence", fontweight="bold")
plt.show()


**Interpretation.** Calendar-month means do not align across the two observed years, while neighboring weekly counts are strongly related (lag-1 autocorrelation 0.76), unlike most week-to-week changes. The short window does not rule out seasonality, and level shifts can create persistence without proving forecasting skill.


## 5.6 Districts, groups of weeks and spatial concentrations

We now distinguish three units: districts summarized over time, complete Gaza-wide weeks grouped by their recorded activity, and incident coordinates grouped geographically. A cluster here means similarity or proximity under a stated algorithm, not a verified cause or legal pattern.


### Figure M12 - District counts, typical volume and variability

Top: the existing district-week focal-count heatmap, with the partial week greyed out and excluded from its color scale. Bottom: the existing median-count versus coefficient-of-variation plot, using complete weeks only; the coefficient is SD/mean and bubble area indicates the complete-week total. The district contribution timeline, whose denominator is all districts' focal events, is in [Appendix A12](#A12---Changing-district-contribution); within-district focal/political shares are in [Appendix A5](#A5---Within-district-focal-prevalence).


In [ ]:
pivot = dwc.pivot(index="week_start", columns="admin2", values="focal_events")
roll = pivot.rolling(4).sum()
share4 = roll.div(roll.sum(axis=1), axis=0).dropna()
order = pivot.sum().sort_values(ascending=False).index.tolist()

prof = dwc.groupby("admin2").focal_events.agg(total="sum", median="median", mean="mean", sd="std", peak="max")
prof["cv"] = prof.sd / prof["mean"]
prof["share_of_known_district_focal"] = prof.total / prof.total.sum()
prof["weeks_with_zero"] = dwc.groupby("admin2").focal_events.apply(lambda s: int((s == 0).sum()))
prof = prof.loc[order]
display(prof.round({"mean": 1, "sd": 1, "cv": 2, "share_of_known_district_focal": 3}))
print("Range of each district's 4-week share of known-district focal events:")
display(pd.DataFrame({"min": share4.min(), "max": share4.max()}).loc[order].round(3))

dist_colors = dict(zip(order, ["#8D3B42", "#315A7D", "#E0A458", "#6B8F71", "#9AA5B1"]))

def draw_district_variability(fig, ax):
    ax.scatter(prof["median"], prof["cv"], s=prof.total / 6, color=[dist_colors[d] for d in prof.index], alpha=0.9, edgecolor="black")
    for d, r in prof.iterrows():
        ax.annotate(d, (r["median"], r["cv"]), textcoords="offset points", xytext=(9, 7), fontsize=9)
    ax.set_xlim(0, prof["median"].max() * 1.3); ax.set_ylim(0, prof.cv.max() * 1.2)
    ax.set_xlabel("Median focal events per week (typical volume)"); ax.set_ylabel("Variability (sd / mean of weekly counts)")
    ax.set_title("Typical volume vs. burstiness by district (bubble = total)")
    ax.grid(axis="x", visible=True)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.stackplot(share4.index, [share4[d] for d in order], labels=order, colors=[dist_colors[d] for d in order], alpha=0.9)
ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.set_xlim(share4.index.min(), share4.index.max())
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8.5)
ax.set_xlabel("Week start (4-week window ending that week)"); ax.set_ylabel("Share of focal events with a known district")
ax.set_title("Which districts carry the focal events, and does the split change over time?")
ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.30), fontsize=8.5)
plt.tight_layout(); fig.subplots_adjust(bottom=0.30); save_supporting_figure("A12", fig)


In [ ]:
fig = plt.figure(figsize=(15, 9), layout="constrained")
grid = fig.add_gridspec(2, 1, height_ratios=[1, 1.2])
draw_district_heatmap(fig, fig.add_subplot(grid[0]))
draw_district_variability(fig, fig.add_subplot(grid[1]))
fig.suptitle("Figure M12. District-week counts and typical volume versus variability", fontweight="bold")
plt.show()


**Interpretation.** Gaza City has the largest observed district-week peak and contributes 3,411 complete-week focal records (32.3% of the known-district complete-week total), while North Gaza has the highest relative variability (0.83). The common early-2025 trough and different district volumes support local baselines, with partial-week and complete-week scopes kept distinct.


### Grouping complete weeks by recorded activity

**Source:** `acled_gaza_week.csv`, 103 complete weeks. We create **ln(1 + focal count)** and **ln(1 + political count)** here in Phase 3 to reduce the influence of large counts, then combine them with the unlogged focal share. These weekly logs are not Phase 2's `fatalities_log1p` variable.

**StandardScaler** subtracts each feature's mean and divides by its standard deviation so different units do not dominate distance. **K-means** groups weeks with similar standardized values; it is fitted to the full historical period only for description, and the count/share features overlap mathematically. We compare k=2-5 before describing three groups.


In [ ]:
feat = gwc[["focal_events", "political_events", "focal_share"]].copy()
feat["focal_events"], feat["political_events"] = np.log1p(feat.focal_events), np.log1p(feat.political_events)
Xs = StandardScaler().fit_transform(feat)

sil = []
for k in range(2, 6):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Xs)
    sil.append({"k": k, "silhouette": silhouette_score(Xs, km.labels_), "smallest cluster (weeks)": int(np.bincount(km.labels_).min())})
display(pd.DataFrame(sil).set_index("k").round(3))


**Choosing a descriptive grouping.** The silhouette score compares how similar a week is to its assigned group versus other groups; higher values indicate clearer separation for these features. Although k=2 scores highest (0.67), k=3 (0.51) is retained to describe low weeks and distinguish the remaining activity/share patterns, not to claim a true number of regimes.


### Figure A10 - Groups of complete weeks and their timing

The scatter shows raw focal/political counts for readability, but group assignments use the transformed three-feature input described above. Group A/B/C names follow increasing median focal count; the timeline shows whether similar weeks occur together.


In [ ]:
K = 3
km = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit(Xs)
gwc["cluster"] = km.labels_
# Name clusters by their median focal count so the labels are stable between runs
rank = gwc.groupby("cluster").focal_events.median().sort_values().index.tolist()
names = {c: f"Group {chr(65 + i)}" for i, c in enumerate(rank)}
gwc["regime"] = gwc.cluster.map(names)
cl_prof = gwc.reset_index().groupby("regime").agg(weeks=("focal_events", "size"), first=("week_start", "min"), last=("week_start", "max"),
                                                  median_focal=("focal_events", "median"), median_political=("political_events", "median"),
                                                  median_share=("focal_share", "median"))
display(cl_prof.round({"median_focal": 1, "median_political": 1, "median_share": 3}))
print("Week starts per group:")
for g, d in gwc.groupby("regime"):
    print(f"  {g}:", ", ".join(f"{x:%Y-%m-%d}" for x in d.index) if len(d) <= 12 else f"{len(d)} weeks; first {d.index.min():%Y-%m-%d}, last {d.index.max():%Y-%m-%d}")

pal = dict(zip(sorted(names.values()), [C_GREY, C_OTHER, C_FOCAL]))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.9), gridspec_kw={"width_ratios": [1, 1.5]})
ax = axes[0]
for g, d in gwc.groupby("regime"):
    ax.scatter(d.political_events, d.focal_events, color=pal[g], s=36, alpha=0.9, label=f"{g} ({len(d)} weeks)")
ax.set_xlabel("Political-violence records per week"); ax.set_ylabel("Focal events per week")
ax.set_title("Which kinds of weeks group together?"); ax.legend(frameon=False); ax.grid(axis="x", visible=True)
ax = axes[1]
ax.plot(gwc.index, gwc.focal_events, color="black", lw=0.7, alpha=0.5)
for g, d in gwc.groupby("regime"):
    ax.scatter(d.index, d.focal_events, color=pal[g], s=36, label=g, zorder=3)
ax.set_xlabel("Week start"); ax.set_ylabel("Focal events per week")
ax.set_title("When does each kind of week occur?"); ax.legend(frameon=False, ncol=3)
plt.tight_layout(); save_supporting_figure("A10", fig)


The supporting figure and its interpretation are in [Appendix A10](#A10---Exploratory-groups-of-weeks); the calculations and tables above remain available here.


### Spatial input and concentration tables

**Source:** `acled_focal_incidents.csv`, post-cutoff land focal incidents. We count incidents at each coordinate pair and repeat the calculation for precision 1; top-ten/top-twenty lists are ranked separately within each subset. District percentages below use land focal incidents, not the district-week denominator; the accompanying Gini and Lorenz comparison uses each subset's own occupied coordinates.


In [ ]:
def site_table(df):
    return (df.groupby(["latitude", "longitude"])
              .agg(events=("event_id_cnty", "size"), district=("admin2", lambda s: s.mode().iat[0]),
                   location=("location", lambda s: s.mode().iat[0]), precision1_share=("geo_precision_1", "mean"))
              .reset_index().sort_values("events", ascending=False))

sites_full = site_table(incidents)
inc_p1 = incidents[incidents.geo_precision_1]
sites_p1 = site_table(inc_p1)

def gini(v):
    v = np.sort(np.asarray(v, float)); n = len(v)
    return (2 * np.arange(1, n + 1) - n - 1).dot(v) / (n * v.sum())

conc = pd.DataFrame({
    "Incidents": [len(incidents), len(inc_p1)],
    "Distinct coordinate pairs": [len(sites_full), len(sites_p1)],
    "Median incidents per coordinate": [sites_full.events.median(), sites_p1.events.median()],
    "Top 10 coordinates, share of incidents": [sites_full.events.head(10).sum() / len(incidents), sites_p1.events.head(10).sum() / len(inc_p1)],
    "Top 20 coordinates, share of incidents": [sites_full.events.head(20).sum() / len(incidents), sites_p1.events.head(20).sum() / len(inc_p1)],
    "Gini of incidents across coordinates": [gini(sites_full.events), gini(sites_p1.events)],
}, index=["All land focal incidents", "Precision 1 only"]).T
display(conc.round(3))
print("Share of incidents by district:")
display(incidents.admin2.value_counts(normalize=True).round(3).rename("share").to_frame())
print("Ten largest coordinates (all land focal incidents):")
display(sites_full.head(10).round({"precision1_share": 3}))


### Spatial-map inputs

**Source:** the land focal-incident CSV, full and precision-1 subsets. The existing maps are combined with the concentration curves in [Figure M13](#Figure-M13---Spatial-concentration-and-precision-sensitivity); they use common extent, bubble scaling and color normalization, without a basemap or inferred boundaries.


In [ ]:
def draw_spatial_maps(fig, axes):
    vmax = sites_full.events.max()
    for ax, sites, title in [(axes[0], sites_full, f"All land focal incidents ({len(incidents):,})"),
                             (axes[1], sites_p1, f"Precision 1 only ({len(inc_p1):,})")]:
        s = sites.sort_values("events")
        sc = ax.scatter(s.longitude, s.latitude, s=6 + s.events * 0.3, c=s.events, cmap="Reds", norm=LogNorm(vmin=1, vmax=vmax),
                        edgecolor="black", linewidth=0.4, alpha=0.85)
        ax.set_title(title); ax.set_xlabel("Longitude")
        ax.set_aspect(1 / np.cos(np.radians(31.45))); ax.grid(True)
        for _, r in sites.head(4).iterrows():
            ax.annotate(r.location, (r.longitude, r.latitude), textcoords="offset points", xytext=(10, 8), fontsize=7.5,
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8),
                        arrowprops=dict(arrowstyle="-", lw=0.6, color="black"))
    axes[0].set_ylabel("Latitude")
    cb = fig.colorbar(sc, ax=axes, shrink=0.75, pad=0.02); cb.set_label("Incidents at the coordinate (log scale)")


The spatial tables record 10,583 incidents on 101 coordinate pairs, versus 7,576 on 95 pairs under precision 1. [Figure M13](#Figure-M13---Spatial-concentration-and-precision-sensitivity) combines the geographic and concentration views, while [Appendix A13](#A13---Largest-coordinates-and-recording-precision) preserves the top-coordinate precision bars.


### Figure M13 - Spatial concentration and precision sensitivity

Top: the existing full and precision-1 land focal maps, with identical coordinate scales and color/size definitions. Bottom: the existing Lorenz curves, accumulating incidents from least to most populated coordinates; the diagonal represents equal counts and Gini summarizes inequality. Each subset uses its own occupied-coordinate set, so this compares selected populations rather than a fixed set of locations; [Appendix A13](#A13---Largest-coordinates-and-recording-precision) shows precision composition at the largest coordinates.


In [ ]:
top12 = sites_full.head(12).iloc[::-1].copy()
top12["p1_events"] = (top12.events * top12.precision1_share).round()
top12["other_events"] = top12.events - top12.p1_events


def draw_lorenz(fig, ax):
    for sites, lab, col in [(sites_full, "All land focal incidents", C_FOCAL), (sites_p1, "Precision 1 only", C_OTHER)]:
        v = np.sort(sites.events.to_numpy())
        ax.plot(np.r_[0, np.arange(1, len(v) + 1) / len(v)], np.r_[0, v.cumsum() / v.sum()], color=col, lw=2, label=f"{lab} (Gini {gini(v):.2f})")
    ax.plot([0, 1], [0, 1], color="black", ls="--", lw=1, label="Even spread")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Share of coordinates (smallest to largest)"); ax.set_ylabel("Cumulative share of incidents")
    ax.set_title("How unevenly are incidents spread across coordinates?"); ax.legend(frameon=False, loc="upper left"); ax.grid(True)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top12.location, top12.p1_events, color=C_FOCAL, label="Precision 1")
ax.barh(top12.location, top12.other_events, left=top12.p1_events, color=C_GREY, label="Precision 2-3")
for y, (_, r) in enumerate(top12.iterrows()):
    ax.text(r.events + 6, y, f"{int(r.events)}  ({r.precision1_share:.0%} precision 1)", va="center", fontsize=8.5)
ax.set_xlim(0, top12.events.max() * 1.4)
ax.set_xlabel("Land focal incidents at the coordinate"); ax.set_title("Which coordinates hold the most incidents, and how precise are they?")
ax.legend(frameon=False, loc="lower right"); ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); save_supporting_figure("A13", fig)


In [ ]:
fig = plt.figure(figsize=(15, 11), layout="constrained")
grid = fig.add_gridspec(2, 2, height_ratios=[1.6, 1])
map_full = fig.add_subplot(grid[0, 0])
map_p1 = fig.add_subplot(grid[0, 1], sharex=map_full, sharey=map_full)
maps = [map_full, map_p1]
draw_spatial_maps(fig, maps)
draw_lorenz(fig, fig.add_subplot(grid[1, :]))
fig.suptitle("Figure M13. Land focal locations and precision sensitivity", fontweight="bold")
plt.show()


**Interpretation.** The broad spatial shape persists after precision filtering, but the top-ten share rises from 33.8% to 38.1% and Gini from 0.57 to 0.60 while about 28% of incidents are excluded. Concentration therefore depends on which records and places remain, and shared named-place coordinates do not identify exact incident points.


### Figure M14 - How sensitive is spatial clustering to its settings?

**Source:** full and precision-1 spatial incidents, including repeated coordinate rows. **DBSCAN** connects dense nearby observations: radius (`eps`) sets neighborhood distance and `min_samples` sets how many observations are needed for a dense neighborhood; isolated observations are called noise. Distances use haversine on radians with radius converted from kilometers, and the grid compares 0.25-3 km with min_samples 10/50 ([method reference](#Spatial-clustering-method)).

The table reports clusters, unclustered share and the largest cluster's share of all incidents. These are descriptive sensitivity measures, not predictive scores.


In [ ]:
R_EARTH_KM = 6371.0088
def run_dbscan(df, eps_km, min_samples):
    X = np.radians(df[["latitude", "longitude"]].to_numpy())          # geographic coordinates in radians
    db = DBSCAN(eps=eps_km / R_EARTH_KM, min_samples=min_samples, metric="haversine", algorithm="ball_tree").fit(X)
    return db.labels_

rows = []
for subset, d in [("All land focal incidents", incidents), ("Precision 1 only", inc_p1)]:
    for ms in (10, 50):
        for eps in (0.25, 0.5, 1.0, 1.5, 2.0, 3.0):
            lab = run_dbscan(d, eps, ms)
            n_cl = len(set(lab)) - (1 if -1 in lab else 0)
            counts = pd.Series(lab[lab >= 0]).value_counts()
            rows.append({"subset": subset, "min_samples": ms, "eps_km": eps, "clusters": n_cl,
                         "noise_share": (lab == -1).mean(), "largest_cluster_share": counts.max() / len(d) if len(counts) else 0.0})
db_grid = pd.DataFrame(rows)
display(db_grid.round({"noise_share": 3, "largest_cluster_share": 3}))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
styles = {("All land focal incidents", 10): (C_FOCAL, "-"), ("All land focal incidents", 50): (C_FOCAL, "--"),
          ("Precision 1 only", 10): (C_OTHER, "-"), ("Precision 1 only", 50): (C_OTHER, "--")}
for ax, col, ylab, title in [(axes[0], "clusters", "Clusters found", "How many clusters at each radius?"),
                             (axes[1], "noise_share", "Share of incidents left as noise", "How much is left unclustered?"),
                             (axes[2], "largest_cluster_share", "Share of incidents in the largest cluster", "When does everything merge?")]:
    for (subset, ms), (c, ls) in styles.items():
        d = db_grid[(db_grid.subset == subset) & (db_grid.min_samples == ms)]
        ax.plot(d.eps_km, d[col], color=c, ls=ls, marker="o", label=f"{subset.replace(' land focal incidents', '')}, min_samples={ms}")
    if col != "clusters":
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("DBSCAN radius (km, haversine)"); ax.set_ylabel(ylab); ax.set_title(title, fontsize=10)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()


**Interpretation.** With min_samples=10, the full subset has 81 clusters at both 0.25 and 0.5 km, but their noise shares differ; cluster counts fall to 19 at 2 km and one at 3 km. Repeated named-place coordinates strongly shape this result, so 1-2 km is a range worth investigating against district summaries and precision restrictions, not an established optimal setting.


# Task 6 - Answers to the business questions

Each answer below names its population and links to the exact evidence in this notebook. Counts describe this historical ACLED snapshot; they are not a complete measure of civilian harm.


## BQ1 - How do counts and shares differ between periods?

**Answer.** Among Gaza political-violence records, focal share is **16.2% before October 7, 2023 (63/389)** and **47.6% from that date (10,651/22,364)**. Adjusting each total for the 644/725 calendar days gives **0.68 versus 102.8 focal events per seven days**, rather than a mean of weekly CSV rows.

**Evidence:** [period table](#2.1-Before-and-from-October-7:-counts,-shares-and-rates) and [Figure M8](#Figure-M8---Period-rates,-focal-shares-and-within-type-evidence) for counts, shares and rates; [Task 4, Hypothesis A results](#4.2-Hypothesis-A---Is-focal-share-different-between-periods?) for the test/effect sizes. Figure M8 includes the intervals and within-type comparison.

**Meaning and limit.** The increase also appears within Explosions, so it is not solely a change in event-type mix; the sparse earlier cohort and changing recording coverage prevent a causal explanation.


## BQ2 - Which actor labels and event types recur?

**Answer.** Among **10,714 focal records across both periods**, Explosions/Remote violence accounts for **9,906 (92.5%)** and Violence against civilians for **808 (7.5%)**. Air/drone strike is the largest sub-event type (**8,420; 78.6%**), followed by shelling/artillery/missile attack (**1,442**) and Attack (**792**); Civilians (Palestine) appears as actor2 in every focal record, while Military Forces of Israel (2022-) appears as actor1 in **10,493 (97.9%)**.

**Evidence:** [Task 2, Figure M6 and focal cross-tabs](#2.5-Which-actor-labels-and-sub-event-types-occur-together?) for event/sub-event combinations and actor counts; [actor tables](#1.4-Recorded-actor-labels) and [Appendix A1](#A1---Recorded-actor-profiles) for actor marginals. These percentages describe composition **within focal events**, not the focal prevalence within each event type shown in [Figure M9](#4.3-Hypothesis-B---Does-focal-share-differ-by-event-type?).

**Meaning and limit.** The civilian label is required by the focal rule and one actor label dominates; assigning roles beyond the recorded labels still requires incident-note verification.


## BQ3 - Where are land focal events concentrated?

**Answer.** Among **10,583 post-cutoff land focal incidents**, Gaza City accounts for **32.5%**, Khan Yunis **21.5%**, and Deir El Balah **21.4%**. The top ten coordinate pairs hold **33.8%** of incidents; after restricting to precision 1, the top ten within that subset hold **38.1%**, but only **7,576 incidents remain** (about 28% excluded).

**Evidence:** [Task 5 spatial concentration tables](#Spatial-input-and-concentration-tables) for district/top-ten percentages and retained counts; [Figure M13, map panels](#Figure-M13---Spatial-concentration-and-precision-sensitivity) for the full/restricted maps; [Figure M13, concentration panel](#Figure-M13---Spatial-concentration-and-precision-sensitivity) and [Appendix A13](#A13---Largest-coordinates-and-recording-precision) for concentration and uneven precision.

**Meaning and limit.** The broad geography persists but concentration changes with selection; repeated named-place coordinates are not exact attack points, and temporal stability of spatial clusters has not been established.


## BQ4 - Which district-weeks have high or sustained activity?

**Answer.** The Gaza-wide peak is **254 focal events** in the week starting **October 28, 2023**; the largest district-week is **Gaza City, November 4, 2023, with 102**. Gaza City totals **3,451 focal events across all 104 weeks**, or **3,411 across complete weeks only**; Gaza-wide focal counts also correlate with the previous week (**Spearman 0.70**) and preceding four-week total (**0.66**).

Relative to recent **eligible local history**, **48 of 465 scored district-weeks (10.3%)** exceed the screening threshold. The largest upward score is Khan Yunis on March 29, 2025 (**48 focal events**); several other high scores follow the early-2025 trough, illustrating how a recovery can trigger a short-window alert.

**Evidence:** [Figure M10](#5.3-Which-Gaza-wide-weeks-stand-out?) for the Gaza peak; [district tables](#2.7-District-week-counts) and [Figure M12](#Figure-M12---District-counts,-typical-volume-and-variability) for all-week totals/peak; [Figure M12](#Figure-M12---District-counts,-typical-volume-and-variability) for complete-week totals; [Figure M7, left](#2.10-Which-measures-move-together,-and-which-categories-are-associated?) for Spearman history relationships; [Task 5 local-history tables](#District-week-screening-and-baseline-sensitivity) for the 48/465 flags and named district-weeks.

**Meaning and limit.** Absolute peaks and unusual local weeks are different questions, and these screening flags are leads for investigation rather than verified anomaly labels.


## BQ5 - Which findings change under restrictions or settings?

**Answer.** Precision materially changes focal prevalence: the later-period share is **59.6% at precision 1 versus 31.7% at precision 2**, compared with **21.7% versus 14.2%** before the cutoff. Explosions remains the dominant type among focal events (**61.9% before; 92.6% after**), and Gaza City has the most later-period focal events within both precision categories; before the cutoff, North Gaza has **28** focal records versus Gaza City's **10**, from a total of only 63.

Spatial top-ten concentration rises from **33.8% to 38.1%** under precision 1; DBSCAN merges from many clusters into one as radius increases, while changing local-history eligibility changes district-week flags from **50/475 to 48/465**. These comparisons show that definitions and settings affect what stands out.

**Evidence:** [Figure M5 and precision tables](#2.4-Does-location-precision-change-the-focal-share?) for precision-specific prevalence; [BQ5 supporting tables](#Supporting-tables-for-BQ5) below for focal composition and district rankings; [spatial tables](#Spatial-input-and-concentration-tables) for top-ten shares; [Figure M14 and grid](#Figure-M14---How-sensitive-is-spatial-clustering-to-its-settings?) for spatial settings; [local-history comparison](#District-week-screening-and-baseline-sensitivity) for baseline eligibility.

**Limit.** These checks do not fully test alternative focal definitions, reporting coverage, geographic aggregation or temporal spatial stability; those remain explicit follow-up questions.


### Supporting tables for BQ5

These tables display the focal-composition and district-ranking claims directly, rather than citing a precision-share plot that uses a different denominator. **Source:** `acled_events_clean.csv`; the labels specify the period, focal restriction and precision subset.


In [ ]:
# Direct supporting evidence for BQ5; all values are computed from the event master.
focal_all = master.loc[master.focal_event]
focal_type_period = pd.crosstab(focal_all.period, focal_all.event_type).reindex([BEFORE, AFTER], fill_value=0)
print("Focal-event composition by period (denominator: all focal events in that period)")
display(focal_type_period)
display(focal_type_period.div(focal_type_period.sum(axis=1), axis=0).mul(100).round(1).add_suffix(" (%)"))
post_focal_evidence = focal_all.loc[focal_all.period.eq(AFTER) & focal_all.geo_precision.isin([1, 2])]
print("Post-cutoff focal counts by known district and precision")
display(pd.crosstab(post_focal_evidence.admin2, post_focal_evidence.geo_precision))
print("Pre-cutoff focal counts by known district")
display(focal_all.loc[focal_all.period.eq(BEFORE)].groupby("admin2").size().rename("Focal events").to_frame())


# Task 7 - Key insights and modeling plan

## Seven insights to carry into Phase 4

1. **Account for unequal periods and a sparse earlier cohort.** The period contrast is large, but only 63 earlier focal records support it; use it as descriptive context, with temporal modeling based on post-cutoff history ([period table](#2.1-Before-and-from-October-7:-counts,-shares-and-rates), [M8](#Figure-M8---Period-rates,-focal-shares-and-within-type-evidence)).
2. **Separate definition from discovery.** Actor and event-type patterns partly reflect the focal rule; retain original labels for explanation, not as proof of roles or a shortcut target ([M6](#2.5-Which-actor-labels-and-sub-event-types-occur-together?), [M9](#4.3-Hypothesis-B---Does-focal-share-differ-by-event-type?)).
3. **Retain and review the fatality tail.** Most records report zero fatalities, while flagged high totals contribute disproportionately; use robust summaries and source verification rather than automatic deletion ([M1](#1.2-Reported-fatalities---common-values-and-rare-extremes), [A7](#A7---Highest-reported-fatality-records), [A8](#A8---Timing-of-high-fatality-flags)).
4. **Carry precision sensitivity into spatial work.** The restriction removes about 28% of land focal incidents and changes concentration, so present full and restricted results together ([M13](#Figure-M13---Spatial-concentration-and-precision-sensitivity)).
5. **Use local history and test recovery effects.** Counts persist over time, but an eight-week baseline can flag recovery from a trough; compare longer windows and preserve eligibility rules ([M10](#5.3-Which-Gaza-wide-weeks-stand-out?), [local checks](#District-week-screening-and-baseline-sensitivity), [M11](#Figure-M11---Calendar-patterns-versus-persistence)).
6. **Use district-specific comparisons and both counts and shares.** Districts differ in volume/variability, and a high share can occur with few records; retain denominators with alerts ([M10](#5.3-Which-Gaza-wide-weeks-stand-out?), [A5](#A5---Within-district-focal-prevalence), [M12](#Figure-M12---District-counts,-typical-volume-and-variability)).
7. **Treat spatial clusters as concentrations of recorded places.** Repeated coordinates and radius choices strongly determine cluster size; compare DBSCAN against simple district totals and investigate stability ([spatial tables](#Spatial-input-and-concentration-tables), [M14](#Figure-M14---How-sensitive-is-spatial-clustering-to-its-settings?)).

## Proposed techniques and evaluation

The two selected technique families are **spatial clustering** and **temporal anomaly detection**, continuing the Phase 1 plan within Phase 2's revised Gaza/post-cutoff scope. The work below is a plan, not completed predictive validation.

| Technique | Inputs and purpose | Baseline / comparisons | How to judge usefulness |
|---|---|---|---|
| DBSCAN | Post-cutoff land focal coordinates in radians; identify geographic concentrations for documentary review | District-count summaries; radii around 1-2 km plus the wider explored range, min_samples 10/50, full versus precision 1 | Cluster/noise proportions, sensitivity of membership and concentration, source review of selected places; useful only if it adds stable detail beyond district totals |
| District-week anomaly detection | Assess a completed week's focal activity against eligible earlier weeks; optionally add political counts, focal share, lag1 and previous-four-week sum for Isolation Forest | Trailing median/MAD as the transparent baseline; compare windows such as 8/12/16 weeks and an Isolation Forest alternative | Alert burden per district, consistency across settings, review of high/low alerts and unflagged comparison weeks, and sensitivity to recovery/zero-denominator history |

**Timing and leakage.** The proposed alert is retrospective, produced after the current week closes, not a forecast of next week's harm. Fit any learned model or transformation on earlier weeks only and tune settings using rolling chronological validation within the post-cutoff period; full-period EDA and K-means here have already seen the snapshot, so none of it is an untouched final test set. Future newly collected data would be needed for a genuinely unseen assessment.

**Feature choices.** Prior-week and prior-four-week counts are motivated by persistence, while district, denominators and precision are needed to interpret coverage. Counts and their ratios share information, so compare smaller feature sets rather than assuming each is independent; do not use record IDs, source-update timestamps or the review flag as direct predictors of an unverified target. Calendar effects remain a sensitivity question rather than a demonstrated seasonal mechanism.

**No verified anomaly labels.** Rule-based flags are not ground truth for evaluating another detector, and accuracy against those flags would be circular. Start with alert burden, stability and documented review; calculate precision/recall only if an independent review process supplies suitable labels, and describe that process.

**First experiment.** Compare eligible district-specific trailing baselines at 8, 12 and 16 weeks, recording which alerts are trough recoveries and how results change with history eligibility; in parallel, compare spatial settings with district totals and the precision-1 subset. Proceed to a more complex method only if it adds interpretable, stable review candidates that the simple baseline misses, with each candidate linked to event IDs, notes and sources.


# Limitations and reproducibility

- **Recorded coverage:** ACLED records are not all events or all civilian harm; changes in documentation can change counts and shares.
- **Meaning of fields:** fatality totals belong to entire event records; actor positions, targeting labels and precision codes do not independently verify roles, intent or exact locations.
- **Dependence and selection:** related events limit independence-based tests; precision restrictions change the observed population, and categorical associations can be partly built into the focal rule.
- **Screening, not ground truth:** anomaly flags, K-means groups and DBSCAN clusters depend on settings and have not been validated as important events. The short historical window does not rule out seasonality.
- **What remains untested:** verified actor roles, reporting-bias effects, alternative cohort definitions, and stability across time/geographic aggregation require further analysis and source review.

To reproduce the analysis, start from the project folder and run the notebook from top to bottom using the supplied four CSVs. The initial checks stop on snapshot mismatches; randomized procedures use seed 42, Monte Carlo uses 99,999 resamples, and incomplete weeks are excluded from scoring. No notebook cell writes back to a source CSV.

The data source is the ACLED Palestine event export downloaded October 18, 2026, with a raw-file manifest in `data/metadata/`. Phase 2's country/area filter, retained blanks and derived features are summarized in the [data guide](#Data-guide---what-is-being-counted?); runtime versions are printed below.


In [ ]:
import platform
from importlib.metadata import version
print("Python", platform.python_version())
for package in ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "scikit-learn"]:
    print(package, version(package))


# Appendix - Supporting figures

These existing plots provide optional detail for the main report. They retain their original populations, calculations and plot types; the source links return to the accompanying calculations and tables. [Return to the main figure register](#Task-3---Main-figure-register).


## A1 - Recorded actor profiles

Source, population and method: [original analysis](#1.4-Recorded-actor-labels).


In [ ]:
show_supporting_figure("A1")


**Interpretation.** Actor2 is blank in 32.4% of all records, but every focal record lists Civilians (Palestine) there, while Military Forces of Israel (2022-) appears as actor1 in 97.9% of focal records. The civilian label is part of the focal rule and the Israeli-forces labels change by date, so these are descriptions of coded records rather than independently verified roles.


## A2 - All-event district and month counts

Source, population and method: [original analysis](#1.6-Where-and-when-all-events-are-recorded).


In [ ]:
show_supporting_figure("A2")


**Interpretation.** Gaza City has the most records (8,036), and monthly recording increases markedly around October 2023, with a visible trough in February 2025. The pattern motivates period and district comparisons, but recorded volume can reflect reporting coverage as well as activity and does not explain why a change happened.


## A3 - Fatality ECDF by focal status

Source, population and method: [original analysis](#2.2-Reported-fatalities-by-focal-status).


In [ ]:
show_supporting_figure("A3")


**Interpretation.** About 31% of focal records report zero fatalities, compared with 86% of other political-violence records, and the medians are 2 and 0. This compares reported event totals, not civilian deaths, and a recorded zero may also mean no fatality figure was available.


## A4 - Event-type composition by period

Source, population and method: [original analysis](#2.6-Did-the-mix-of-event-types-change?).


In [ ]:
show_supporting_figure("A4")


**Interpretation.** Explosions/Remote violence rises from 77% to 89% of political-violence records, while Battles falls from 14% to 7%. Because the mix changes, the overall focal-share difference needs both this composition view and the within-type test rather than either alone.


## A5 - Within-district focal prevalence

Source, population and method: [original analysis](#2.9-How-does-focal-share-vary-within-districts?).


In [ ]:
show_supporting_figure("A5")


**Interpretation.** Across complete weeks, pooled focal share ranges from 40.4% in North Gaza to 52.1% in Deir El Balah, while Gaza City is 50.0% (3,411/6,825). Weekly shares fluctuate much more than the four-week pooled line, especially with small denominators, and North Gaza has two complete weeks with undefined shares.


## A6 - Smaller-class sample sizes

Source, population and method: [original analysis](#5.1-Are-some-groups-too-small-for-reliable-comparison?).


In [ ]:
show_supporting_figure("A6")


**Interpretation.** The full political-record split is nearly balanced, but the earlier period has only 63 focal examples, later Violence against civilians has only six non-focal examples, and later Battles has no focal examples. These structural and small-sample limitations matter more than the overall balance; North Gaza also has zero-count district-weeks, where a political denominator of zero leaves focal share undefined.


## A7 - Highest reported fatality records

Source, population and method: [original analysis](#5.2-Which-high-fatality-records-need-review?).


In [ ]:
show_supporting_figure("A7")


**Interpretation.** The largest recorded total is 471 and the three largest records fall in October-November 2023, while 422 of the 576 flagged records are focal. Large totals may reflect how reports were aggregated as well as event severity, so retain them and review their notes and sources rather than treating them as automatically wrong.


## A8 - Timing of high-fatality flags

Source, population and method: [original analysis](#Figure-A8---When-are-flagged-records-concentrated?).


In [ ]:
show_supporting_figure("A8")


**Interpretation.** Q4 2023 contains 281 of the 576 flagged records, or 7.7% of that quarter's records, compared with 2.6% in Q1 2024 and 1.7% or less in later quarters. Sensitivity analyses that exclude flagged records would therefore affect the early period disproportionately; the flag is a review threshold, not verification of the totals.


## A9 - Monthly grouped count/share summary

Source, population and method: [original analysis](#5.4-Is-there-a-longer-term-direction?).


In [ ]:
show_supporting_figure("A9")


**Interpretation.** Focal counts have little monotonic trend over complete weeks (Spearman rho=0.10), while political counts decrease (rho=-0.55) and focal share increases (rho=0.51). The series contains a marked trough and recovery rather than a smooth trend, and the reported exploratory p-values do not account for dependence between neighboring weeks.


## A10 - Exploratory groups of weeks

Source, population and method: [original analysis](#Grouping-complete-weeks-by-recorded-activity).


In [ ]:
show_supporting_figure("A10")


**Interpretation.** The three descriptive groups contain eight low weeks from January 18-March 8, 2025, 51 lower-share weeks, and 44 higher-share weeks, with median focal counts of 21.5, 92 and 123.5 respectively. Their strong temporal ordering suggests level shifts, but overlapping count/share information and the chosen scaling mean these are exploratory labels, not validated regimes or predictions.


## A11 - Focal records by weekday

Source and method: [original analysis](#Calendar-comparison-inputs).


In [ ]:
show_supporting_figure("A11")


**Interpretation.** The busiest weekday has about 1.11 times the count of the quietest. Counts are not adjusted for weekday exposure, so this display does not establish a weekday effect.


## A12 - Changing district contribution

Source and method: [original analysis](#Figure-M12---District-counts,-typical-volume-and-variability).


In [ ]:
show_supporting_figure("A12")


**Interpretation.** These shares describe each district’s contribution to all known-district focal events over four weeks, rather than focal/political prevalence within a district. The changing mix adds geographic context to the main district overview.


## A13 - Largest coordinates and recording precision

Source and method: [original analysis](#Figure-M13---Spatial-concentration-and-precision-sensitivity).


In [ ]:
show_supporting_figure("A13")


**Interpretation.** The precision-1 proportion varies substantially across the largest recorded places, so the restriction removes incidents unevenly. These are counts at recorded coordinate pairs, often shared named places, rather than verified exact incident points.


# References

1. **ACLED.** Palestine event-level export, January 1, 2022-September 30, 2025, downloaded October 18, 2026; raw file `data/raw/acled_palestine_events_2022_2025.csv` and manifest `data/metadata/acled_events_download_manifest.json`. [Codebook](https://acleddata.com/methodology/acled-codebook) provides source definitions; use of the data remains subject to ACLED's terms.
2. **Group 7.** `DMA_Phase1.ipynb`, business problem and original BQ1-BQ5; `DMA_Phase2.ipynb`, revised Gaza scope, preprocessing and four processed outputs. This report summarizes the dependencies needed to read Phase 3 independently.
3. **DS312.** `[STUDENT] DS312_FINAL_PIT_GUIDE.pdf`, Phase 3 requirements (page 5), 1st Semester A.Y. 2026-2027, Instructor Nicole S. Menorias; `Phase3_EDA_Guide_G7.pdf`, supporting course guide.
4. **Group 7 contributions.** Univariate, bivariate/multivariate, hypothesis-testing, pattern-spotting and BQ-answer work integrated here; `DMA_Phase3_EDA_Bivariate_Multivariate_Merged.ipynb` was consulted as a read-only comparison reference.
5. **Mugot, C. J. S.** *EDA Bivariate Analysis of Categorical Variables*, 2026, course reference for categorical analysis/Monte Carlo testing.


### Outlier screening method

6. **Iglewicz, B. and Hoaglin, D. (1993).** *How to Detect and Handle Outliers*. Modified z-score screening reference; the eight-week window and count MAD floor are project choices, not universal thresholds.


### Spatial clustering method

7. **Ester, M., Kriegel, H.-P., Sander, J. and Xu, X. (1996).** *A density-based algorithm for discovering clusters in large spatial databases with noise*. DBSCAN method reference.
8. **Software references.** SciPy: `chi2_contingency`, `MonteCarloMethod`, `fisher_exact`, `binomtest`, `spearmanr`; scikit-learn: `DBSCAN`, `KMeans`, `StandardScaler`, `silhouette_score`. The printed runtime versions identify the implementation used for this notebook's saved run.
